# Adaptive path Correction with Exponents (ACE)
Demo code to reproduce the results and figures in the paper. A single A6000 GPU was used to run the demo notebook. Run the code blocks in order to prevent errors.

## Step 1. Import libraries and train models (if not already trained)

In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import torch
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import itertools
import json
from IPython.display import clear_output

# Import ACE library components
from ace_lib.metrics.export import compute_sample_based_metrics
from ace_lib.interpolant import MLPInstFlexible
from ace_lib.sample_data import ground_truth_hcg, plot_diagnostics
from ace_lib.ace import simulate_ace
from ace_lib.interpolant import Interpolant, FlowMatcher, plot_path_trajectories
from ace_lib.utils import load_interpolants_from_json, set_seed, get_experiment_dir

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
set_seed(42)

interpolant_schedules = load_interpolants_from_json("ace_lib/interpolant_schedules.json")

Skip the pretraining code if the 2D models are already trained under `/PretrainedToyModels/`

In [ ]:
# Train toy models once (if not already trained) / Takes ~30 mins
from ace_lib.train_toy_models import main
main()

## Figure 1. Marginal Path Collapse and Our Solution ACE 
$q^{(1)}q^{(2)}/q^{(3)}$

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_1")
print(f"Experiment directory: {exp_dir}")

In [ ]:
schedule_combination = ["cos_t", "ddpm_linear", "default_linear"]

# Load the pretrained models
model_path = "PretrainedToyModels"
u_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); u_model1.load_state_dict(torch.load(f"{model_path}/u_model1_X_given_A_alpha={schedule_combination[0]}.pth")); u_model1.eval()
s_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); s_model1.load_state_dict(torch.load(f"{model_path}/s_model1_X_given_A_alpha={schedule_combination[0]}.pth")); s_model1.eval()
u_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); u_model2.load_state_dict(torch.load(f"{model_path}/u_model2_XY_given_B_alpha={schedule_combination[1]}.pth")); u_model2.eval()
s_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); s_model2.load_state_dict(torch.load(f"{model_path}/s_model2_XY_given_B_alpha={schedule_combination[1]}.pth")); s_model2.eval()
u_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); u_model3.load_state_dict(torch.load(f"{model_path}/u_model3_X_alpha={schedule_combination[2]}.pth")); u_model3.eval()
s_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); s_model3.load_state_dict(torch.load(f"{model_path}/s_model3_X_alpha={schedule_combination[2]}.pth")); s_model3.eval()

def v1_fn(x, t, A): return u_model1(x, t, A)
def s1_fn(x, t, A): return s_model1(x, t, A)
def v2_fn(z, t, B): return u_model2(z, t, B)
def s2_fn(z, t, B): return s_model2(z, t, B)
def v3_fn(x, t): return u_model3(x, t)
def s3_fn(x, t): return s_model3(x, t)
def sigma_fn(t): return 0.5 * torch.ones_like(t)

print("Models loaded.")

In [ ]:
# Define the velocity, score, projection, embedding lists
A, B = 1, 1
v_fn_list=[
        lambda x, t: v1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # v1(X|A)
        lambda x, t: v2_fn(x, t, torch.full((x.size(0), 1), B, device=x.device)), # v2(X|B)
        lambda x, t: v3_fn(x[:, :1], t)                                                         # v3(Z)
    ]
s_fn_list=[
        lambda x, t: s1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # s1(X|A)
        lambda x, t: s2_fn(x, t, torch.full((x.size(0), 1), B, device=x.device)), # s2(X|B)
        lambda x, t: s3_fn(x[:, :1], t)                                                         # s3(Z)
    ]
proj_list=[
        lambda z: z[:, :1],    # project to X 
        lambda z: z,           # identity for Z
        lambda z: z[:, :1]     # project to X
    ]
emb_list=[
        lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
        lambda z: z,                                                                  # identity
        lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1)   # embed X→Z
    ]
print("Velocity, score, projection, and embedding functions defined.")

In [ ]:
Bump=0.0
Ramp=0.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
print(f"Gamma and dGamma functions defined with Bump={Bump}_Ramp={Ramp}_weight={weight}.")

In [ ]:
# Criterion plot
Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])
t = torch.linspace(0.0, 1.0, 100)
plt.plot(t.numpy(), Criterion(t).numpy())
plt.xlabel('t')
plt.ylabel('Criterion(t)')
plt.ylim(-20, 100)
plt.title('Criterion vs t')
plt.grid(True)
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}.png"))
plt.show()

# print when Criterion = 0
for i in range(len(t)-1):
    if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
        print("Criterion = 0 at t =", t[i].item())
        break
for i in range(len(t)-1):
    if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
        print("Criterion = 0 at t =", t[i].item())
        break

In [ ]:
A, B = 1, 1
x0 = torch.randn(10000, 2).to("cuda")  # (X, Y) sample

samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
    x0=x0, 
    v_fn_list=v_fn_list, 
    s_fn_list=s_fn_list, 
    proj_list=proj_list, 
    emb_list=emb_list, 
    sigma_fn=sigma_fn,
    v_star= lambda z, t: v2_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
    t0=0.0, t1=1.0, n_steps=100, device="cuda", ess_threshold=0.4, print_resample_history=True,
    gamma_list=gamma_list,
    d_gamma_list=d_gamma_list,
    resample=False
    )
print("ACE simulation completed.")
samples = samples.cpu().numpy()
plot_diagnostics(samples, logw_final, logw_history, save_name=f"{exp_dir}/diagnostic_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}")
plt.close()

In [ ]:
plot_path_trajectories(sample_history, n_frame=4, experiment_id=exp_dir, name=f"trajectory_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}_NR", deg=-50)

In [ ]:
A, B = 1, 1
x0 = torch.randn(10000, 2).to("cuda")  # (X, Y) sample

samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
    x0=x0, 
    v_fn_list=v_fn_list, 
    s_fn_list=s_fn_list, 
    proj_list=proj_list, 
    emb_list=emb_list, 
    sigma_fn=sigma_fn,
    v_star= lambda z, t: v2_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
    t0=0.0, t1=1.0, n_steps=100, device="cuda", ess_threshold=0.4, print_resample_history=True,
    gamma_list=gamma_list,
    d_gamma_list=d_gamma_list,
    resample=True
    )
print("ACE simulation completed.")
samples = samples.cpu().numpy()
plot_diagnostics(samples, logw_final, logw_history, save_name=f"{exp_dir}/diagnostic_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}_FKC")
plt.close()

In [ ]:
plot_path_trajectories(sample_history, n_frame=4, experiment_id=exp_dir, name=f"trajectory_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}_FKC", deg=-50)

In [ ]:
print(schedule_combination)

In [ ]:
Bump=10.0
Ramp=2.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
print(f"Gamma and dGamma functions defined with Bump={Bump}_Ramp={Ramp}_weight={weight}.")

In [ ]:
# Criterion plot
Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])
t = torch.linspace(0.0, 1.0, 2000)
plt.plot(t.numpy(), Criterion(t).numpy())
plt.xlabel('t')
plt.ylabel('Criterion(t)')
plt.ylim(-20, 100)
plt.title('Criterion vs t')
plt.grid(True)
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}.png"))
plt.show()

# print when Criterion = 0
for i in range(len(t)-1):
    if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
        print("Criterion = 0 at t =", t[i].item())
        break
for i in range(len(t)-1):
    if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
        print("Criterion = 0 at t =", t[i].item())
        break

In [ ]:
Bump_Values = [0.0, 10.0, 20.0, 30.0, 50.0, 100.0]
Ramp_Values = [0.0, 0.5, 1.0, 1.5, 2.0]

for Bump in Bump_Values:
    for Ramp in Ramp_Values:
        print(f"\nAnalyzing for Bump={Bump}, Ramp={Ramp}")
        weight = 1.0

        # Define the exponent list
        gamma_list = [
            lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
            lambda t : torch.tensor(1 * weight),
            lambda t : torch.tensor(-1 * weight)
        ]
        d_gamma_list = [
            lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
            lambda t: torch.zeros_like(t),
            lambda t: torch.zeros_like(t)
        ]
        Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])
        t = torch.linspace(0.0, 1.0, 2000)
        for i in range(len(t)-1):
            if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
                print("Criterion = 0 at t =", t[i].item())
                break
        for i in range(len(t)-1):
            if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
                print("Criterion = 0 at t =", t[i].item())
                break
        

In [ ]:
A, B = 1, 1
x0 = torch.randn(10000, 2).to("cuda")  # (X, Y) sample

samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
    x0=x0, 
    v_fn_list=v_fn_list, 
    s_fn_list=s_fn_list, 
    proj_list=proj_list, 
    emb_list=emb_list, 
    sigma_fn=sigma_fn,
    v_star= lambda z, t: v2_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
    t0=0.0, t1=1.0, n_steps=100, device="cuda", ess_threshold=0.4, print_resample_history=True,
    gamma_list=gamma_list,
    d_gamma_list=d_gamma_list,
    resample=True
    )
print("ACE simulation completed.")
samples = samples.cpu().numpy()
plot_diagnostics(samples, logw_final, logw_history, save_name=f"{exp_dir}/diagnostic_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}_ACE")
plt.close()

In [ ]:
plot_path_trajectories(sample_history, n_frame=4, experiment_id=exp_dir, name=f"trajectory_plot_Bump={Bump}_Ramp={Ramp}_weight={weight}_ACE", deg=-50)

In [ ]:
Bump=0.0
Ramp=0.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
print(f"Gamma and dGamma functions defined with Bump={Bump}_Ramp={Ramp}_weight={weight}.")

Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])

t = torch.linspace(0.0, 1.0, 100)
plt.plot(t.numpy(), Criterion(t).numpy(), label="Constant Exponents")
plt.xlabel('t')
plt.ylabel('C(t)')
plt.grid(True)


Bump=10.0
Ramp=5.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
print(f"Gamma and dGamma functions defined with Bump={Bump}_Ramp={Ramp}_weight={weight}.")

Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])

t = torch.linspace(0.0, 1.0, 2000)
plt.plot(t.numpy(), Criterion(t).numpy(), label='Adaptive Exponents (with Bump)')
plt.legend()
plt.ylim((-50,100))
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_{schedule_combination}_Bump{Bump}.png"))
plt.show()

# print when Criterion = 0
for i in range(len(t)-1):
    if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
        print("Criterion(t): pos -> neg at t =", t[i].item())
        break
for i in range(len(t)-1):
    if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
        print("Criterion(t) neg -> pos at t =", t[i].item())
        break

## Figure 2. Non-integrable Region in the ratio-of-Gaussians example

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_2")
print(f"Experiment directory: {exp_dir}")

We can simplify the GMM example for convenience:

Let $p_t$ denote the path from $p_0=\mathcal{N}(0,\sigma_0^2I)$ to $p_1=\mathcal{N}(0,\sigma_1^2I)$ via $X_t = (1-t)X_0 + tX_1$. 

Then, the intermediate densities will be $p_t = \mathcal{N}(0, \left((1-t)^2\sigma_0^2 + t^2 \sigma_1^2\right)I)$

The score function is simply
$$
\nabla \log p_t (X) = -\frac{1}{(1-t)^2\sigma_0^2 + t^2 \sigma_1^2} X
$$

Deriving the SDE from the Path

For a general SDE of the form $dX_t = f(t,X_t)dt + g(t)dW_t$, the evolution of its variance $\sigma_t^2$ for a zero-mean Guassian process is governed by the Fokker-Planck equation:
$$
\frac{d\sigma_t^2}{dt} = 2\mu(t)\sigma_t^2 + g(t)^2
$$
where we assumed linear drift of the form $f(t,X_t) = \mu(t)X_t$. For this specific path, we have
$$
\mu(t) = \frac{-2(1-t)\sigma_0^2 + 2t\sigma_1^2 - g(t)^2}{2((1-t)^2\sigma_0^2 + t^2\sigma_1^2)}
$$
and the full drift is then $f(t,X_t) = \mu(t)X_t$.

As a result, we have
$$
dX_t = \frac{-2(1-t)\sigma_0^2 + 2t\sigma_1^2 - g(t)^2}{2((1-t)^2\sigma_0^2 + t^2\sigma_1^2)} X_tdt + g(t)dW_t
$$

In [ ]:
BUMP_VALUE = 0.0

def sigma_p1(t): return ((1-t)**2 + 0.5*t**2)/(1 + BUMP_VALUE * t * (1-t))
def sigma_p2(t): return (1-t)**2 + 7*t**2
def sigma_q1(t): return 1.5*(1-t)**2 + t**2
def sigma_q2(t): return 1.5*(1-t)**2 + t**2

def sigma_eff(var1, var2):
    return 1.0 / (1/var1 + 1/var2)

def sigma_P(t):
    return sigma_eff(sigma_p1(t), sigma_p2(t))

def sigma_Q(t):
    return sigma_eff(sigma_q1(t), sigma_q2(t))

def eff_var(ts, var1, var2, filter_negative=False):
    if filter_negative:
        indices = np.where(var1 < var2)[0]
    else:
        indices = np.arange(len(ts))
    eff_ts = ts[indices]
    eff_vars = 1.0 / (1/var1[indices] - 1/var2[indices])
    return eff_ts, eff_vars

# Find break points where integrability flips
ts = np.linspace(0,1,500)
sigP = np.array([sigma_P(t) for t in ts])
sigQ = np.array([sigma_Q(t) for t in ts])
diff = sigP - sigQ
sign_changes = ts[np.where(np.diff(np.sign(diff)))[0]]
eff_ts, eff_vars = eff_var(ts, sigP, sigQ)

# Generate 10000 samples * 500 timesteps from Gaussian with effective variance 
samples = np.random.randn(len(eff_ts), 10000, 2) * np.sqrt(eff_vars[:, np.newaxis, np.newaxis])
print("Samples shape:", samples.shape)
samples = torch.tensor(samples, device="cuda", dtype=torch.float32)
plot_path_trajectories(samples, n_frame=6, resample_history=None, divergence_points=[0.456, 0.63],  experiment_id=exp_dir, name=f"gaussian_ratio_path_Bump={BUMP_VALUE}", deg=-45, num_trajectory_points=0, hard_lim=15)

print("Potential divergence points at:", sign_changes)
t_eff = 0.456
i_eff = (np.abs(eff_ts - t_eff)).argmin()
print(f"Effective variance at {t_eff}: {eff_vars[i_eff]}")

# Plot variances
plt.rcParams['figure.dpi'] = 300
plt.figure(figsize=(8,4))
plt.plot(ts, sigP, label=r"$\sigma_P^2(t)$")
plt.plot(ts, sigQ, label=r"$\sigma_Q^2(t)$")
plt.plot(eff_ts, eff_vars, label=r"$\sigma_{eff}^2(t)$", ls=':')
plt.axhline(0, color='black', lw=0.5)
for sc in sign_changes:
    plt.axvline(sc, color='red', ls='--', label="break point")
plt.legend()
plt.xlabel("t")
plt.ylabel("effective variance")
plt.ylim(-1, 40)
# plt.title("Where integrability breaks (2D Gaussian ratio)")
plt.show()

In [ ]:
bs = 5000
ts = np.linspace(0,1,500)
sigP = np.array([sigma_P(t) for t in ts])
sigQ = np.array([sigma_Q(t) for t in ts])
ts, eff_vars = eff_var(ts, sigP, sigQ, filter_negative=False)

samples = [torch.randn(bs,2) * torch.sqrt(torch.tensor(eff_vars[i])) for i in range(len(ts))]

plot_path_trajectories(samples, divergence_points=[0.456, 0.63], hard_lim=10, experiment_id=".", name="blowup_path", n_frame=5, deg=-80, num_trajectory_points=0, interval_d=50)

In [ ]:
sigma_0 = 1.5
sigma_1 = 1

n_samples = 5000
dim = 2
n_timesteps = 1000
T = 1.0
dt = T / n_timesteps

# For any linear path between two isotropic gaussians, the score and velocity functions are:
def v_fn(x, t, sigma_0, sigma_1):
    return (-2 * (1-t) * sigma_0**2 + 2 * t * sigma_1 **2) * x / ( 2 * ((1-t)**2 * sigma_0**2 + t**2 * sigma_1**2) )
def s_fn(x, t, sigma_0, sigma_1):
    return - x / ((1-t)**2 * sigma_0**2 + t**2 * sigma_1**2)
def g_fn(t):
    return 0.5

X_t = torch.randn(n_samples, dim, device=device) * sigma_0
sample_history = [X_t.clone()]

for i in tqdm(range(n_timesteps), desc="Simulating SDE"):
    t = i * dt
    g_t = g_fn(t)
    
    v_value = v_fn(X_t, t, sigma_0, sigma_1) + 0.5 * g_t**2 * s_fn(X_t, t, sigma_0, sigma_1)
    drift = v_value * dt

    random_noise = torch.randn_like(X_t)
    diffusion = g_t * np.sqrt(dt) * random_noise
    
    X_t = X_t + drift + diffusion
    
    sample_history.append(X_t.clone())

print(f"Simulation finished. `sample_history` contains {len(sample_history)} timesteps.")
# plot_path_trajectories(sample_history, resample_history=None, hard_lim=10, experiment_id=exp_dir, name="gaussian_path_4")

## Figure E.12: Stabilizing a ratio-of-Gaussians path via the bump parameter

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_E12")
print(f"Experiment directory: {exp_dir}")

In [ ]:
BUMP_VALUE = 0.1 #0.5

def sigma_p1(t): return ((1-t)**2 + 0.5*t**2)/(1 + BUMP_VALUE * t * (1-t))
def sigma_p2(t): return (1-t)**2 + 7*t**2
def sigma_q1(t): return 1.5*(1-t)**2 + t**2
def sigma_q2(t): return 1.5*(1-t)**2 + t**2

def sigma_eff(var1, var2):
    return 1.0 / (1/var1 + 1/var2)

def sigma_P(t):
    return sigma_eff(sigma_p1(t), sigma_p2(t))

def sigma_Q(t):
    return sigma_eff(sigma_q1(t), sigma_q2(t))

def eff_var(ts, var1, var2, filter_negative=False):
    if filter_negative:
        indices = np.where(var1 < var2)[0]
    else:
        indices = np.arange(len(ts))
    eff_ts = ts[indices]
    eff_vars = 1.0 / (1/var1[indices] - 1/var2[indices])
    return eff_ts, eff_vars

# Find break points where integrability flips
ts = np.linspace(0,1,500)
sigP = np.array([sigma_P(t) for t in ts])
sigQ = np.array([sigma_Q(t) for t in ts])
diff = sigP - sigQ
sign_changes = ts[np.where(np.diff(np.sign(diff)))[0]]
eff_ts, eff_vars = eff_var(ts, sigP, sigQ)

# Generate 10000 samples * 500 timesteps from Gaussian with effective variance 
samples = np.random.randn(len(eff_ts), 10000, 2) * np.sqrt(eff_vars[:, np.newaxis, np.newaxis])
print("Samples shape:", samples.shape)
samples = torch.tensor(samples, device="cuda", dtype=torch.float32)
plot_path_trajectories(samples, overwrite_fractions=[0.0, 0.25, 0.5, 0.75, 1.0], experiment_id=exp_dir, name=f"gaussian_ratio_path_Bump={BUMP_VALUE}", deg=-45, num_trajectory_points=0, hard_lim=15)

print("Potential divergence points at:", sign_changes)
t_eff = 0.456
i_eff = (np.abs(eff_ts - t_eff)).argmin()
print(f"Effective variance at {t_eff}: {eff_vars[i_eff]}")

# Plot variances
plt.rcParams['figure.dpi'] = 300
plt.figure(figsize=(8,2))
plt.plot(ts, sigP, label=r"$\sigma_P^2(t)$")
plt.plot(ts, sigQ, label=r"$\sigma_Q^2(t)$")
plt.plot(eff_ts, eff_vars, label=r"$\sigma_{eff}^2(t)$", ls=':')
plt.axhline(0, color='black', lw=0.5)
for sc in sign_changes:
    plt.axvline(sc, color='red', ls='--', label="break point")
plt.legend()
plt.xlabel("t")
plt.ylabel("effective variance")
plt.ylim(-1, 40)
# plt.title("Where integrability breaks (2D Gaussian ratio)")
plt.show()

# print the minimum 1/effective variance
# min_C = 1.0 / np.max(eff_vars[eff_vars > 0])
# print(f"Minimum Criterion min_t C(t): {min_C}")
print(f"C(0.5) = {1.0 / (eff_vars[250])}")

In [ ]:
BUMP_VALUE = 0.4

def sigma_p1(t): return ((1-t)**2 + 0.5*t**2)/(1 + BUMP_VALUE * t * (1-t))
def sigma_p2(t): return (1-t)**2 + 7*t**2
def sigma_q1(t): return 1.5*(1-t)**2 + t**2
def sigma_q2(t): return 1.5*(1-t)**2 + t**2

def sigma_eff(var1, var2):
    return 1.0 / (1/var1 + 1/var2)

def sigma_P(t):
    return sigma_eff(sigma_p1(t), sigma_p2(t))

def sigma_Q(t):
    return sigma_eff(sigma_q1(t), sigma_q2(t))

def eff_var(ts, var1, var2, filter_negative=False):
    if filter_negative:
        indices = np.where(var1 < var2)[0]
    else:
        indices = np.arange(len(ts))
    eff_ts = ts[indices]
    eff_vars = 1.0 / (1/var1[indices] - 1/var2[indices])
    return eff_ts, eff_vars

# Find break points where integrability flips
ts = np.linspace(0,1,500)
sigP = np.array([sigma_P(t) for t in ts])
sigQ = np.array([sigma_Q(t) for t in ts])
diff = sigP - sigQ
sign_changes = ts[np.where(np.diff(np.sign(diff)))[0]]
eff_ts, eff_vars = eff_var(ts, sigP, sigQ)

# Generate 10000 samples * 500 timesteps from Gaussian with effective variance 
samples = np.random.randn(len(eff_ts), 10000, 2) * np.sqrt(eff_vars[:, np.newaxis, np.newaxis])
print("Samples shape:", samples.shape)
samples = torch.tensor(samples, device="cuda", dtype=torch.float32)
plot_path_trajectories(samples, n_frame=6, resample_history=None,  experiment_id=exp_dir, name=f"gaussian_ratio_path_Bump={BUMP_VALUE}", deg=-45, num_trajectory_points=0, hard_lim=15)

print("Potential divergence points at:", sign_changes)
t_eff = 0.456
i_eff = (np.abs(eff_ts - t_eff)).argmin()
print(f"Effective variance at {t_eff}: {eff_vars[i_eff]}")

# Plot variances
plt.rcParams['figure.dpi'] = 300
plt.figure(figsize=(8,4))
plt.plot(ts, sigP, label=r"$\sigma_P^2(t)$")
plt.plot(ts, sigQ, label=r"$\sigma_Q^2(t)$")
plt.plot(eff_ts, eff_vars, label=r"$\sigma_{eff}^2(t)$", ls=':')
plt.axhline(0, color='black', lw=0.5)
for sc in sign_changes:
    plt.axvline(sc, color='red', ls='--', label="break point")
plt.legend()
plt.xlabel("t")
plt.ylabel("effective variance")
plt.ylim(-1, 40)
# plt.title("Where integrability breaks (2D Gaussian ratio)")
plt.show()

## Figure 3. Common noise schedules and Marginal Path Collapse

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_3")
print(f"Experiment directory: {exp_dir}")

In [ ]:
names = list(interpolant_schedules.keys())

t = torch.linspace(0.0, 1.0, 100)

name_eq = {
    "custom_poly": r"$\alpha_t = -4t^3+7t^2-4t+1$",
    "ddpm_linear": r"$\alpha_t = \text{DDPM}$",
    "1-t**2": r"$\alpha_t = 1-t^2$",
    "sigmoid": r"$\alpha_t = \text{Sigmoid}$",
    "default_linear": r"$\alpha_t = 1-t$",
    "cos_t": r"$\alpha_t=\cos(\frac{\pi}{2}t)$"
}


plt.figure(figsize=(7, 7)) 
for name in names:
    plt.plot(t.numpy(), interpolant_schedules[name].alpha_t(t).numpy(), label=name_eq[name])
    # print(name, interpolant_schedules[name].alpha_t(0), interpolant_schedules[name].beta_t(0), interpolant_schedules[name].d_alpha_t(0), interpolant_schedules[name].d_beta_t(0))
plt.xlabel(r'$t$')
plt.ylabel(r'$\alpha_t$')
# plt.title(r'$\alpha_t$ vs $t$ Graph for Common Noise Schedules')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(exp_dir, "alpha_t_plot_common_schedules.png"))
plt.show()

In [ ]:
#tau_t = sqrt(1- interpolant_schedules["sigmoid"].alpha_t(t))
tau = lambda t: torch.sqrt(1 - interpolant_schedules["sigmoid"].alpha_t(t))

plt.figure(figsize=(7, 7))
plt.plot(t.numpy(), tau(t).numpy(), label=r"$\tau(t)$")
plt.plot(t.numpy(), interpolant_schedules["1-t**2"].alpha_t(tau(t)).numpy(), label=r"$1-\tau^2(t)$")
plt.plot(t.numpy(), interpolant_schedules["sigmoid"].alpha_t(t).numpy(), label=r"sigmoid$(t)$", ls='--')
plt.xlabel(r'$t$')
plt.ylabel(r'$\tau_t$')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(exp_dir, "alpha_t_plot_common_schedules.png"))
plt.show()

In [ ]:
def check_condition(alpha_funcs, g=1.0, n_grid=2000, Bump = 0.0, Ramp = 0.0, NegRamp = 0.0):
    """Check sign conditions for [a1, a2] or [a1, a2, a3]."""
    ts = torch.linspace(0.0, 1.0, n_grid)

    alphas = [f(ts) for f in alpha_funcs]
    Bumps = torch.tensor([Bump * t * (1-t) for t in ts])
    Ramps = torch.tensor([Ramp * t for t in ts])
    NegRamps = torch.tensor([NegRamp * t for t in ts])
    if len(alpha_funcs) == 2:
        C = 2 / (alphas[0]**2 + 1e-12) - g / (alphas[1]**2 + 1e-12) + Bumps / (alphas[0]**2 + 1e-12) + Ramps / (alphas[0]**2 + 1e-12)
    elif len(alpha_funcs) == 3:
        if interpolant_schedules["cos_t"].alpha_t in alpha_funcs:
            # find the index of cos_t
            cos_index = alpha_funcs.index(interpolant_schedules["cos_t"].alpha_t)
            # put cos_t at the beginning
            if cos_index != 0:
                alphas[0], alphas[cos_index] = alphas[cos_index], alphas[0]
                print(f"Swapped cos_t {cos_index} to the front for calculation.")
        C = 1 / (alphas[0]**2 + 1e-12) + 1 / (alphas[1]**2 + 1e-12) - (g - NegRamps) / (alphas[2]**2 + 1e-12) + Bumps / (alphas[0]**2 + 1e-12) + Ramps / (alphas[0]**2 + 1e-12)
    else:
        raise ValueError("Only supports 2 or 3 schedules")

    return (C[0] > 0) and (C.min() < 0), ts, C


name_eq_plot = {
    "custom_poly": r"$-4t^3+7t^2-4t+1$",
    "ddpm_linear": r"$\text{DDPM}$",
    "1-t**2": r"$1-t^2$",
    "sigmoid": r"$\text{Sigmoid}$",
    "default_linear": r"$1-t$",
    "cos_t": r"$\cos(\frac{\pi}{2}t)$"
}


def find_valid_combinations(interpolants, g=1.0, Bump=0.0, Ramp=0.0, NegRamp=0.0):
    valid_pairs, valid_triples = [], []
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['font.size'] = 14

    for a1, a2, a3 in reversed(list(itertools.permutations(interpolants, 3))):
        # plot 5 combinations (to avoid clutter)
        if len(valid_triples) > 5:
            break

        if "custom_poly" in [a1, a2, a3]:
            continue

        valid, t, C = check_condition([interpolant_schedules[a1].alpha_t, interpolant_schedules[a2].alpha_t, interpolant_schedules[a3].alpha_t], g=g, Bump=Bump, Ramp=Ramp, NegRamp=NegRamp)
        if valid:
            valid_triples.append([a1, a2, a3])
            plt.ylim((-20,100))
            plt.grid(True)
            plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
            plt.legend()
    plt.xlabel('t')
    plt.ylabel(r"$C(t)$")
    plt.savefig(os.path.join(exp_dir, f"Criterion_plot_a1a2a3_no_bump.png"))
    plt.show()
    plt.close()

    if len(valid_triples) == 0:
        for a1, a2 in itertools.permutations(interpolants, 2):
            valid, t, C = check_condition([interpolants[a1].alpha_t, interpolants[a2].alpha_t], g=g, Bump=Bump, Ramp=Ramp)
            if valid:
                valid_pairs.append([a1, a1, a2])
                plt.ylim((-20,100))
                plt.grid(True)
                plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a1]}, {name_eq_plot[a2]}')
                plt.legend()
        plt.xlabel('t')
        plt.ylabel(r"$C(t)$")
        plt.savefig(os.path.join(exp_dir, f"Criterion_plot_a1a1a2_no_bump.png"))
        plt.show()
        plt.close()

    return valid_pairs, valid_triples

In [ ]:
# Visualize randomly selected triples that violate the criterion (just 6 for clean plots; full search in appendix)
triples = [
    ["cos_t", "ddpm_linear", "default_linear"],
    ["1-t**2", "cos_t", "ddpm_linear"],
    ["sigmoid", "cos_t", "ddpm_linear"],
    ["cos_t", "cos_t", "sigmoid"],
    ["sigmoid", "cos_t", "default_linear"],
    ["cos_t", "cos_t", "default_linear"]
]

for a1, a2, a3 in triples:
    valid, t, C = check_condition([interpolant_schedules[a1].alpha_t, interpolant_schedules[a2].alpha_t, interpolant_schedules[a3].alpha_t], g=1.0, 
                                  Bump=0.0, Ramp=0.0, NegRamp=0.0)
    plt.ylim((-20,100))
    plt.grid(True)
    plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
    plt.legend()
plt.xlabel('t')
plt.ylabel(r"$C(t)$")
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_a1a2a3_with_bump.png"))
plt.show()
plt.close()

for a1, a2, a3 in triples:
    valid, t, C = check_condition([interpolant_schedules[a1].alpha_t, interpolant_schedules[a2].alpha_t, interpolant_schedules[a3].alpha_t], g=1.0, 
                                  Bump=10.0, Ramp=2.0, NegRamp=0.0)
    plt.ylim((-20,100))
    plt.grid(True)
    plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
    plt.legend()
plt.xlabel('t')
plt.ylabel(r"$C(t)$")
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_a1a2a3_with_bump.png"))
plt.show()
plt.close()


## Figure 4. Visualization of the sampling trajectories

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_4")
print(f"Experiment directory: {exp_dir}")

In [ ]:
Bump=0.0
Ramp=2.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
A, B = 1, 1
x0 = torch.randn(10000, 2).to("cuda")  # (X, Y) sample

samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
    x0=x0, 
    v_fn_list=v_fn_list, 
    s_fn_list=s_fn_list, 
    proj_list=proj_list, 
    emb_list=emb_list, 
    sigma_fn=sigma_fn,
    v_star= lambda z, t: v2_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
    t0=0.0, t1=1.0, n_steps=100, device="cuda", ess_threshold=0.7, print_resample_history=True,
    gamma_list=gamma_list,
    d_gamma_list=d_gamma_list,
    resample=True
    )
print("ACE simulation completed.")
samples = samples.cpu().numpy()
plot_path_trajectories(method_figure=True, sample_history=sample_history, resample_history=resample_history, n_frame=0, experiment_id=exp_dir, name=f"effect_of_resampling_Bump={Bump}_Ramp={Ramp}_weight={weight}_ACE", deg=-50)
plt.close()

In [ ]:
Bump=0.0
Ramp=2.0
weight = 1.0

# Define the exponent list
gamma_list = [
    lambda t : torch.tensor(1) + Bump * t * (1 - t) + Ramp * t,
    lambda t : torch.tensor(1 * weight),
    lambda t : torch.tensor(-1 * weight)
]
d_gamma_list = [
    lambda t: torch.zeros_like(t) + Bump * (1 - 2 * t) + Ramp,
    lambda t: torch.zeros_like(t),
    lambda t: torch.zeros_like(t)
]
A, B = 1, 1
x0 = torch.randn(10000, 2).to("cuda")  # (X, Y) sample

samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
    x0=x0, 
    v_fn_list=v_fn_list, 
    s_fn_list=s_fn_list, 
    proj_list=proj_list, 
    emb_list=emb_list, 
    sigma_fn=sigma_fn,
    v_star= lambda z, t: v2_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
    t0=0.0, t1=1.0, n_steps=100, device="cuda", ess_threshold=0.7, print_resample_history=True,
    gamma_list=gamma_list,
    d_gamma_list=d_gamma_list,
    resample=False
    )
print("ACE simulation completed.")
samples = samples.cpu().numpy()
plot_path_trajectories(method_figure=False, sample_history=sample_history, resample_history=resample_history, n_frame=6, experiment_id=exp_dir, name=f"effect_of_no_resampling_Bump={Bump}_Ramp={Ramp}_weight={weight}_ACE", deg=-50)
plt.close()

## Table 2. Distributional similarity metrics (lower is better)

Run `ace_eval_script.py`.

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "Table_2")
print(f"Experiment directory: {exp_dir}")

In [ ]:
def compute_summary_stats(input_file="results.csv", output_file="summary_stats.csv"):
    if not os.path.exists(input_file):
        print(f"Error: File '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    group_cols = ['Method', 'Bump', 'Ramp', 'weight']
    metric_cols = ['W1', 'W2', 'MMD', 'TV']

    summary = df.groupby(group_cols)[metric_cols].agg(['mean', 'std', 'min', 'max'])

    # Change ('W1', 'mean') to 'W1_mean'
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    
    summary = summary.reset_index()
    summary.to_csv(output_file, index=False)
    
    print(f"Summary statistics saved to '{output_file}'")
    
    # Optional: Print a preview to console
    print("\n--- Preview of Results ---")
    print(summary.head().to_string())

compute_summary_stats()

In [ ]:
input_file = "summary_stats.csv"
if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Please run summarize_results.py first.")

# 1. Load Data
df = pd.read_csv(input_file)

# 2. Define the exact rows we want in the table
# Added "and Ramp == 0.0" to all ACE rows for safety as discussed
row_definitions = [
    ("NR*",            "Method == 'NR'"),
    ("FKC*",           "Method == 'FKC'"),
    ("ACE ($B=0$)",     "Method == 'ACE' and Bump == 0.0 and Ramp == 1.5"),
    ("ACE ($B=10$)",   "Method == 'ACE' and Bump == 10.0 and Ramp == 1.5"),
    ("ACE ($B=20$)",   "Method == 'ACE' and Bump == 20.0 and Ramp == 1.5"),
    ("ACE ($B=30$)",   "Method == 'ACE' and Bump == 30.0 and Ramp == 1.5"),
    ("ACE ($B=40$)",   "Method == 'ACE' and Bump == 40.0 and Ramp == 1.5"),
    ("ACE ($B=50$)",   "Method == 'ACE' and Bump == 50.0 and Ramp == 1.5"),
    # ("ACE ($B=100$)", "Method == 'ACE' and Bump == 100.0 and Ramp == 2.0"),
]

# 3. Define Metrics and their precision
metrics = [
    ("W1", 2),
    ("W2", 2),
    ("MMD", 3)
]

# 4. Extract data and find "Best" (Minimum Mean) for bolding
table_rows = []

# Initialize minimums to infinity
best_means = {m: float('inf') for m, _ in metrics}

for label, query in row_definitions:
    try:
        subset = df.query(query)
        
        if subset.empty:
            table_rows.append({'label': label, 'data': None})
            continue

        row_data = subset.iloc[0]
        
        processed_row = {'label': label, 'data': {}}
        
        for metric, _ in metrics:
            mean_val = row_data[f"{metric}_mean"]
            
            # Check for global minimum (Best)
            if mean_val < best_means[metric]:
                best_means[metric] = mean_val
            
            processed_row['data'][metric] = {
                'max':  row_data[f"{metric}_max"],
                'mean': mean_val,
                'std':  row_data[f"{metric}_std"]
            }
        
        table_rows.append(processed_row)

    except Exception as e:
        print(f"Error processing row '{label}': {e}")

# 5. Generate LaTeX
print(r"\begin{tabular}{lccccccc c}")
print(r"\toprule")
print(r"\multirow{2}{*}{Method}")
print(r"% & \multirow{2}{*}{Path Validity}")
print(r"& \multicolumn{2}{c}{$W_1$ ($\downarrow$)} ")
print(r"& \multicolumn{2}{c}{$W_2$ ($\downarrow$)} ")
print(r"& \multicolumn{2}{c}{MMD (RBF) ($\downarrow$)} \\")
print(r"% & \multirow{2}{*}{Exponent Schedule} \\")
print(r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}")

# --- CHANGED ORDER HERE ---
print(r"& Mean $\pm$ Std & Max ")
print(r"& Mean $\pm$ Std & Max ")
print(r"& Mean $\pm$ Std & Max \\") 
# --------------------------

print(r"\midrule")

for i, row in enumerate(table_rows):
    label = row['label']
    
    # Add midrule between baselines (FKC) and ACE methods if needed
    # Note: Depending on your table logic, you might want the line after ACE* (B=0) or B=10
    if label == "ACE* ($B=10$)":
        print(r"\midrule")

    if row['data'] is None:
        print(f"{label} & - & - & - & - & - & - \\\\")
        continue

    line_parts = [label]

    for metric, precision in metrics:
        vals = row['data'][metric]

        max_str = f"{vals['max']:.{precision}f}"
        mean_str = f"{vals['mean']:.{precision}f}"
        std_str = f"{vals['std']:.{precision}f}"

        if abs(vals['mean'] - best_means[metric]) < 1e-9:
            mean_str = f"\\textbf{{{mean_str}}}"

        # --- CHANGED ORDER HERE ---
        # First: Mean +/- Std
        line_parts.append(f"{mean_str} $\\pm$ {std_str}")
        # Second: Max
        line_parts.append(f"{max_str}")
        # --------------------------

    line_str = " & ".join(line_parts) + r" \\"
    print(line_str)

print(r"\bottomrule")
print(r"\end{tabular}")

In [ ]:
def generate_sensitivity_grid():
    input_file = "summary_stats.csv"
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    df = pd.read_csv(input_file)

    # 1. Filter for ACE only
    df = df[df['Method'] == 'ACE']

    target_metrics = ['W1', 'W2', 'MMD'] 

    for metric in target_metrics:
        print(f"\n% --- Metric: {metric} ---")
        
        # 2. Pivot
        mean_pivot = df.pivot(index='Bump', columns='Ramp', values=f'{metric}_mean')
        std_pivot = df.pivot(index='Bump', columns='Ramp', values=f'{metric}_std')

        # 3. FILTERING LOGIC (The Fix)
        # We want to keep ALL Rows (Bump), so we drop Columns (Ramp) 
        # that don't have data for every single Bump.
        
        # axis=1 : Look at columns. 
        # how='any': If ANY row in that column is NaN, drop the column.
        mean_pivot.dropna(axis=1, how='any', inplace=True)
        
        # Align std_pivot
        std_pivot = std_pivot.loc[mean_pivot.index, mean_pivot.columns]
        
        if mean_pivot.empty:
            print(f"% WARNING: No common Ramps found across all Bump values for {metric}.")
            continue

        # Sort indices
        mean_pivot.sort_index(axis=0, inplace=True)
        mean_pivot.sort_index(axis=1, inplace=True)
        std_pivot.sort_index(axis=0, inplace=True)
        std_pivot.sort_index(axis=1, inplace=True)

        min_val = mean_pivot.min().min()

        # 4. Generate LaTeX
        num_cols = len(mean_pivot.columns)
        col_format = "l" + "c" * num_cols
        
        print(r"\begin{table}[h]")
        print(r"\centering")
        print(f"\\caption{{ACE Sensitivity Analysis: ${metric}$ ($\\downarrow$)}}")
        
        # Resize to fit linewidth
        print(r"\resizebox{\linewidth}{!}{") 
        print(f"\\begin{{tabular}}{{{col_format}}}")
        print(r"\toprule")
        
        # Header
        headers = [f"{c}" for c in mean_pivot.columns]
        header_str = " & ".join(headers)
        print(f"Bump $\\backslash$ Ramp & {header_str} \\\\")
        print(r"\midrule")

        # Data Rows
        for bump_val in mean_pivot.index:
            row_str = [f"B={bump_val}"]
            
            for ramp_val in mean_pivot.columns:
                m = mean_pivot.loc[bump_val, ramp_val]
                s = std_pivot.loc[bump_val, ramp_val]
                
                prec = 3 if metric == 'MMD' else 2
                m_str = f"{m:.{prec}f}"
                s_str = f"{s:.{prec}f}"
                
                if abs(m - min_val) < 1e-9:
                    m_str = f"\\textbf{{{m_str}}}"
                
                cell = f"{m_str} \\tiny{{$\\pm${s_str}}}"
                row_str.append(cell)

            print(" & ".join(row_str) + r" \\")

        print(r"\bottomrule")
        print(r"\end{tabular}")
        print(r"}") 
        print(r"\end{table}")
        print("\n")

if __name__ == "__main__":
    generate_sensitivity_grid()

In [ ]:
# Visualize linear bump and quadratic bump schedules
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 14
t = torch.linspace(0.0, 1.0, 100)
t_end = 0.8
plt.figure(figsize=(8, 4))
plt.plot(t.numpy(), (t * (1 - t)).numpy(), label=r'$Q(t)=t(1-t)$ (Quadratic Bump)')
plt.plot(t.numpy(), ((t * (t < t_end) + (t_end / (t_end - 1) * (t - t_end) + t_end) * (t >= t_end))).numpy(), label=r'$L_\tau(t)=\min(t, \tau(1-t))$ (Linear Bump)')
plt.plot(t.numpy(), (1.3 * t * (1 - t) + 0.3 * (t * (t < t_end) + (t_end / (t_end - 1) * (t - t_end) + t_end) * (t >= t_end))).numpy(), label=r'$b(t)=B_1Q(t)+B_2L_\tau(t)$ (Combined Bump)')
plt.xlabel(r'$t$')
plt.ylabel('Bump Functions')
plt.legend(loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
def sort_parameter_combinations():
    # 1. Define the search space
    # B1 = Bump (Parabolic term)
    Bump_Values = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 100.0]
    # B2 = Ramp (Linear term)
    Ramp_Values = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 8.0, 16.0]

    combinations = []

    # 2. Iterate through all combinations (Cartesian Product)
    for b1 in Bump_Values:
        for b2 in Ramp_Values:
            # Calculate the cost based on your theorem's integral result
            cost = (1/3) * (b1**2) + (b2**2)
            
            combinations.append({
                "Bump (B1)": b1,
                "Ramp (B2)": b2,
                "Cost": cost
            })

    # 3. Sort by Cost (Ascending)
    df = pd.DataFrame(combinations)
    df = df.sort_values(by="Cost", ascending=True).reset_index(drop=True)

    # 4. Print formatted table
    print(f"{'Rank':<6} | {'Bump (B1)':<10} | {'Ramp (B2)':<10} | {'Cost':<15}")
    print("-" * 50)
    
    for i, row in df.iterrows():
        print(f"{i+1:<6} | {row['Bump (B1)']:<10.1f} | {row['Ramp (B2)']:<10.1f} | {row['Cost']:<15.4f}")

sort_parameter_combinations()

In [ ]:
schedule_combination = ["cos_t", "ddpm_linear", "default_linear"]
# Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[schedule_combination[i]].alpha_t(t))**2 for i in range(len(schedule_combination)) ])

def check_combinations():
    Bump_Values = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 100.0]
    Ramp_Values = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 8.0, 16.0]
    weight = 1.0
    
    # Define time discretization for the check (avoiding t=1 singularity)
    t_eval = torch.linspace(0, 0.999, 1000)

    # --- 2. DEFINE SCHEDULES (ALPHAS) ---
    alpha_funcs = [interpolant_schedules[schedule_combination[i]].alpha_t for i in range(len(schedule_combination))]

    # --- 3. HELPER FUNCTIONS ---
    def get_validity(B, R):
        # 1. Define Gammas (Vectorized)
        # Term 0: Target + Correction
        g0 = 1 + B * t_eval * (1 - t_eval) + R * t_eval
        # Term 1: Conditional
        g1 = torch.full_like(t_eval, 1 * weight)
        # Term 2: Prior (Negative)
        g2 = torch.full_like(t_eval, -1 * weight)
        
        gammas = [g0, g1, g2]
        
        # 2. Compute Criterion
        # C(t) = sum( gamma_i(t) / alpha_i(t)^2 )
        criterion = torch.zeros_like(t_eval)
        for i in range(3):
            alpha = alpha_funcs[i](t_eval)
            criterion += gammas[i] / (alpha**2 + 1e-8) # epsilon for stability
            
        # 3. Check Condition
        min_val = criterion.min().item()
        return "O" if min_val > 0 else "X"

    # --- 4. GENERATE & SORT DATA ---
    results = []
    
    for b in Bump_Values:
        for r in Ramp_Values:
            # Theoretical Smoothness Cost
            cost = (1/3) * (b**2) + (r**2)
            
            # Validity Check
            validity = get_validity(b, r)
            
            results.append({
                "Bump": b,
                "Ramp": r,
                "Cost": cost,
                "Valid": validity
            })

    # Sort by Cost
    df = pd.DataFrame(results)
    df = df.sort_values(by="Cost", ascending=True).reset_index(drop=True)

    # --- 5. PRINT TABLE ---
    print(f"{'Rank':<5} | {'Bump':<6} | {'Ramp':<6} | {'Valid':<5} | {'Cost':<10}")
    print("-" * 45)
    for i, row in df.iterrows():
        print(f"{i+1:<5} | {row['Bump']:<6.1f} | {row['Ramp']:<6.1f} | {row['Valid']:<5} | {row['Cost']:<10.4f}")

check_combinations()

## Table E.5. Frequency of Marginal Path Collapse

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "Table_E5")
print(f"Experiment directory: {exp_dir}")

In [ ]:
def load_interpolants_from_json_alpha_only(path):
    with open(path, "r") as f:
        interpolants_raw = json.load(f)

    interpolants = {}
    for name, funcs in interpolants_raw.items():
        alpha_t = eval(funcs["alpha_t"], {"torch": torch})
        interpolants[name] = alpha_t
    return interpolants

# Load
interpolants = load_interpolants_from_json_alpha_only("ace_lib/interpolant_schedules.json")
names = list(interpolants.keys())

def check_condition(alpha_funcs, n_grid=200, Bump = 0.0, Anneal_weight=1.0):
    """Check sign conditions for [a1, a2, a3]. a1 a3 / a2 and anneal weight applies to (a1 / a2)^w a3"""
    ts = torch.linspace(0.0, 0.99, n_grid)

    if n_grid > 1:
        dt = ts[1] - ts[0]
    else:
        dt = torch.tensor(0.0) 

    alphas = [f(ts) for f in alpha_funcs]
    Bumps = torch.tensor([Bump * t * (1-t) for t in ts])
    if len(alpha_funcs) == 3:
        C = (Anneal_weight + Bumps) / (alphas[0]**2 + 1e-12) - Anneal_weight / (alphas[1]**2 + 1e-12) + 1 / (alphas[2]**2 + 1e-12)
    else:
        raise ValueError("Only supports 3 schedules")

    total_negative_length = 0.0
    if n_grid > 1:
        negative_intervals = C[:-1] < 0
        total_negative_length_tensor = torch.sum(negative_intervals.float()) * dt
        total_negative_length = total_negative_length_tensor.item()

    return (C.min() < 0), ts, C, total_negative_length

name_eq_plot = {
    "ddpm_linear": r"$\text{DDPM}$",
    "1-t**2": r"$1-t^2$",
    "sigmoid": r"$\text{Sigmoid}$",
    "default_linear": r"$1-t$",
    "cos_t": r"$\cos(\frac{\pi}{2}t)$"
}


def find_valid_combinations(interpolants, Anneal_weight=1.0, Bump=0.0, unique=True):
    collapse_combinations = []

    for a1, a2, a3 in itertools.product(interpolants, repeat=3):
        valid, t, C, total_negative_length = check_condition([interpolants[a1], interpolants[a2], interpolants[a3]], Anneal_weight=Anneal_weight, Bump=Bump)
        if valid:
            collapse_combinations.append([a1, a2, a3, total_negative_length])
    collapse_combinations.sort(key=lambda x: x[3], reverse=True)
    if unique:
        unique_combinations = []
        seen = set()
        for combo in collapse_combinations:
            identifier = combo[3]
            if identifier not in seen:
                unique_combinations.append(combo)
                seen.add(identifier)
        collapse_combinations = unique_combinations # Remove duplicates if total_negative_length is the same
    return collapse_combinations

weights = [1.0, 1.1, 1.5, 2.0, 7.5, 15]
for ANNEAL_WEIGHT in weights:
    collapse_combinations = find_valid_combinations(interpolants, Anneal_weight=ANNEAL_WEIGHT, Bump=0.0, unique=False)
    print(f"There are {len(collapse_combinations)} combinations that have path collapse under anneal weight {ANNEAL_WEIGHT}.")

    with open(os.path.join(exp_dir, f"non_unique_collapse_combinations_anneal_weight={ANNEAL_WEIGHT}.json"), "w") as f:
        json.dump(collapse_combinations, f, indent=4)

## Table E.6: Metrics under Homogeneous Composition (collapse duration 0%)

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "Table_E6")
print(f"Experiment directory: {exp_dir}")

In [ ]:
names = ['ddpm_linear', 'ddpm_linear', 'ddpm_linear',0]

bs = 10000; n_steps=1000; seeds = [0,1,2,3,4]
ESS_THRESHOLD = 0.7
ANNEAL_WEIGHT = 1.0
BUMP_VALUE = 0.0
RUN_METRIC_EVAL = True
results = []

u_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); u_model1.load_state_dict(torch.load(f"PretrainedToyModels/u_model1_X_given_A_alpha={names[0]}.pth")); u_model1.eval()
s_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); s_model1.load_state_dict(torch.load(f"PretrainedToyModels/s_model1_X_given_A_alpha={names[0]}.pth")); s_model1.eval()
u_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); u_model2.load_state_dict(torch.load(f"PretrainedToyModels/u_model2_XY_given_B_alpha={names[2]}.pth")); u_model2.eval()
s_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); s_model2.load_state_dict(torch.load(f"PretrainedToyModels/s_model2_XY_given_B_alpha={names[2]}.pth")); s_model2.eval()
u_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); u_model3.load_state_dict(torch.load(f"PretrainedToyModels/u_model3_X_alpha={names[1]}.pth")); u_model3.eval()
s_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); s_model3.load_state_dict(torch.load(f"PretrainedToyModels/s_model3_X_alpha={names[1]}.pth")); s_model3.eval()

def v1_fn(x, t, A): return u_model1(x, t, A)
def s1_fn(x, t, A): return s_model1(x, t, A)
def v2_fn(x, t): return u_model3(x, t)
def s2_fn(x, t): return s_model3(x, t)
def v3_fn(z, t, B): return u_model2(z, t, B)
def s3_fn(z, t, B): return s_model2(z, t, B)
def sigma_fn(t): return 0.5 * torch.ones_like(t)

v_fn_list=[
        lambda x, t: v1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # v1(X|A)
        lambda x, t: v2_fn(x[:, :1], t),                                                 # v2(X)
        lambda x, t: v3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # v3(Z|B)
    ]
s_fn_list=[
        lambda x, t: s1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # s1(X|A)
        lambda x, t: s2_fn(x[:, :1], t),                                                 # s2(X)
        lambda x, t: s3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # s3(Z|B)
    ]
proj_list=[
        lambda z: z[:, :1],    # project to X 
        lambda z: z[:, :1],    # project to X
        lambda z: z            # identity for Z
    ]
emb_list=[
        lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
        lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
        lambda z: z  # identity
    ]
print(f"{names} models loaded.")


for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    print(f"Seed set to {seed}")

    for Method_name in ["NR", "ACE"]: # FKC:
        print(f"Method: {Method_name}")
        if Method_name == "FKC" or Method_name == "NR":
                print("Simulating FKC (Constant Gammas)")
                gamma_list = [
                    lambda t : torch.tensor(1) * ANNEAL_WEIGHT,
                    lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                    lambda t : torch.tensor(1)
                ]
                d_gamma_list = [
                    lambda t: torch.zeros_like(t),
                    lambda t: torch.zeros_like(t),
                    lambda t: torch.zeros_like(t)
                ]
        elif Method_name == "ACE":
            print("Simulating ACE (Adaptive Gammas)")
            gamma_list = [
                lambda t : torch.tensor(1) * ANNEAL_WEIGHT + (BUMP_VALUE * t * (1 - t)),
                lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                lambda t : torch.tensor(1)
            ]
            d_gamma_list = [
                lambda t: torch.zeros_like(t) + (BUMP_VALUE * (1 - 2*t)),
                lambda t: torch.zeros_like(t),
                lambda t: torch.zeros_like(t)
            ]
        if seed == 0:
            Criterion = lambda t: sum([ gamma_list[i](t) / (interpolants[names[i]](t))**2 for i in range(len(names)-1) ])
            t = torch.linspace(0.0, 0.99, 100)
            plt.plot(t.numpy(), Criterion(t).numpy())
            plt.xlabel('t')
            plt.ylabel('Criterion C(t)')
            plt.title('Criterion C(t) vs t')
            plt.grid(True)
            plt.ylim(-20,100)
            plt.savefig(os.path.join(exp_dir, f"Criterion_plot_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUE}_Method={Method_name}.png"))
            plt.show()
            plt.close()

        # print when Criterion = 0
        for i in range(len(t)-1):
            if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
                print("Criterion = 0 at t =", t[i].item())
                break
        for i in range(len(t)-1):
            if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
                print("Criterion = 0 at t =", t[i].item())
                break
        for A, B in [(1,1)]: #[(1,1), (1,0), (0,1), (0,0)]:
            print(f"Conditioning on A={A}, B={B}")

            x0 = torch.randn(bs, 2).to("cuda")
            samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
                x0=x0, v_fn_list=v_fn_list, s_fn_list=s_fn_list, proj_list=proj_list, emb_list=emb_list, sigma_fn=sigma_fn,
                v_star= lambda z, t: v3_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
                t0=0.0, t1=1.0, n_steps=n_steps, device="cuda", ess_threshold=ESS_THRESHOLD, print_resample_history=True,
                gamma_list=gamma_list,
                d_gamma_list=d_gamma_list,
                resample= (Method_name != "NR")
            )
            samples = samples.cpu().numpy()
            if seed == 0:
                plot_diagnostics(samples, logw_final, logw_history, save_name=f"{exp_dir}/alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}")
                plot_path_trajectories(sample_history, n_frame=6, resample_history=None, experiment_id=exp_dir, name=f"alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}", deg=-50)
                plt.close(); clear_output()

            if RUN_METRIC_EVAL:
                print(f"Evaluating {Method_name} for AB={A}{B}")
                samples_gt = ground_truth_hcg(bs, cond_A=A, cond_B=B).numpy()
                w1, w2, mmd_rbf, total_var = compute_sample_based_metrics(
                    torch.tensor(samples_gt), torch.tensor(samples)
                )
                results.append([seed, Method_name, w1, w2, mmd_rbf, total_var, A, B, ESS_THRESHOLD])

                df = pd.DataFrame(results, columns=["seed", "method", "W1", "W2", "MMD_RBF", "Total Var.", "A", "B", "ESS_Threshold"])
                df.to_csv(f"{exp_dir}/experiment_results_numseeds{len(seeds)}_bs{bs}_n_steps{n_steps}_ESS{ESS_THRESHOLD}_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUE}.csv", index=False)

In [ ]:

for Bump in []:
    plt.figure(figsize=(15,12))
    plt.title(f'Criterion C(t) with Bump={Bump} for Invalid Common Cases', fontsize=24)
    for names in collapse_combinations:
        a1, a2, a3, invalid_interval = names
        valid, t, C, invalid_interval = check_condition([interpolants[a1], interpolants[a2], interpolants[a3]], Anneal_weight=1.0, Bump=Bump)
        plt.ylim((-20,100))
        plt.grid(True)
        plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
        plt.legend(loc='upper left', fontsize=12, ncol=3)
    plt.xlabel('t', fontsize=24)
    plt.ylabel(r"$C(t)$", fontsize=24)
    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir, f"Criterion_plot_a1a2a3_bump={Bump}.png"))
    plt.show()
    plt.close()

In [ ]:
import os
import glob
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast

# ==========================================
# CONFIGURATION
# ==========================================
PARENT_DIR = exp_dir
OUTPUT_FILENAME = "Rebuttal_Grid_with_Length.png"
plt.rcParams['savefig.dpi'] = 300 # For saved figure resolution

# Metrics to plot (Columns)
METRICS = ['W1', 'W2', 'MMD_RBF']
METRIC_LABELS = {
    'W1': r'$W_1$ Distance ($\downarrow$)',
    'W2': r'$W_2$ Distance ($\downarrow$)',
    'MMD_RBF': r'MMD-RBF ($\downarrow$)'
}

# LaTeX Mappings for Schedules
NAME_EQ_PLOT = {
    "ddpm_linear": r"$\text{DDPM}$",
    "1-t**2": r"$1-t^2$",
    "sigmoid": r"$\text{Sigmoid}$",
    "default_linear": r"$1-t$",
    "cos_t": r"$\cos(\frac{\pi}{2}t)$"
}

# Methods and Colors (Highlight ACE)
PALETTE = {
    'NR': '#B0B0B0',   # Light Gray
    'FKC': '#696969',  # Dark Gray
    'ACE': '#D62728'   # Bold Red
}

# Order of plotting on X-axis
METHOD_ORDER = ['NR', 'FKC', 'ACE']

# ==========================================
# 1. DATA LOADING
# ==========================================
def load_all_data(parent_dir):
    all_data = []
    
    # 1. ROBUST FOLDER FINDING
    try:
        subdirs = [
            os.path.join(parent_dir, d) for d in os.listdir(parent_dir) 
            if os.path.isdir(os.path.join(parent_dir, d)) and d.endswith("_ESS0.9_Noncollapse")
        ]
    except FileNotFoundError:
        print(f"Error: Parent directory '{parent_dir}' not found.")
        return pd.DataFrame() 
    
    # Sort them based on the float number in the first bracket
    def sort_key(path):
        match = re.search(r"\[([\d\.]+)\]", os.path.basename(path))
        return float(match.group(1)) if match else 0
    
    subdirs.sort(key=sort_key)

    for i, folder_path in enumerate(subdirs):
        folder_name = os.path.basename(folder_path)
        
        # Extract and parse the schedule list for the Case Label
        try:
            # The folder structure is usually: [sort_val]['s1', 's2', 's3', len]_Visualizations
            # We extract the list part between ']' and '_Visualizations'
            raw_list_part = folder_name.split(']')[1].split('_ESS0.9_Noncollapse')[0]
            
            if not raw_list_part.endswith(']'):
                raw_list_part += ']'
            
            # Parse list
            schedule_list = ast.literal_eval(raw_list_part)
            schedules = schedule_list[:3]
            schedule_collapse_length = schedule_list[3] if len(schedule_list) > 3 else "N/A"
            # convert string to float with 1 decimal place
            try:
                schedule_collapse_length = f"{float(schedule_collapse_length)*100:.1f}"
            except:
                schedule_collapse_length = "N/A"
            
            # 1. Map to LaTeX (strip existing $ so we can wrap it uniformly)
            latex_schedules = [NAME_EQ_PLOT.get(s, s).replace('$', '') for s in schedules]
            
            # 2. Apply Reordering Logic (User specified: [2, 0, 1])
            # Original: [s1, s2, s3] -> New: [s3, s1, s2]
            reordered = [latex_schedules[2]] + latex_schedules[:2]
            
            # 3. Format into 3 lines: alpha^(i)_t = val
            formatted_lines = []
            for idx, val in enumerate(reordered):
                # Construct LaTeX string: $\alpha^{(i)}_t = val$
                # Using raw strings to handle backslashes safely
                line = r"$\alpha^{(" + str(idx+1) + r")}_t = " + val + r"$"
                formatted_lines.append(line)
            formatted_lines.append(f"\nCollapse Duration: {schedule_collapse_length}%")
            case_label = "\n".join(formatted_lines)

        except Exception as e:
            print(f"Warning: Could not parse label for {folder_name}, using default. Error: {e}")
            case_label = f"Case {i+1}"

        # 2. ROBUST CSV FINDING
        try:
            files_in_folder = os.listdir(folder_path)
            csv_files = [
                os.path.join(folder_path, f) for f in files_in_folder
                if f.startswith("experiment_results") and f.endswith(".csv")
            ]
        except OSError:
            print(f"Could not access folder: {folder_name}")
            continue

        if not csv_files:
            print(f"Skipping {folder_name}: No results CSV found.")
            continue
            
        # Read Data
        df = pd.read_csv(csv_files[0])
        df['Condition'] = df.apply(lambda row: f"({int(row['A'])},{int(row['B'])})", axis=1)
        df['Case'] = case_label
        df['Case_Index'] = i 
        all_data.append(df)

    if not all_data:
        raise ValueError("No data found! Check your paths.")
        
    return pd.concat(all_data, ignore_index=True)

# ==========================================
# 2. PLOTTING
# ==========================================
def create_facet_plot(df):
    # --- FONT SETTINGS ---
    # Set global font family to serif
    plt.rcParams['font.family'] = 'Times New Roman'
    sns.set_context("paper", font_scale=2)

    cases = df.sort_values('Case_Index')['Case'].unique()
    n_cases = len(cases)
    n_metrics = len(METRICS)
    
    # Increase height per case to accommodate 3 lines of text
    # Width = 5 * n_metrics, Height = 3.5 * n_cases (was 3.0)
    fig, axes = plt.subplots(n_cases, n_metrics, figsize=(5 * n_metrics, 3.5 * n_cases), sharex=True)
    
    sns.set_style("whitegrid")

    print("Generating Plots...")

    for row_idx, case in enumerate(cases):
        for col_idx, metric in enumerate(METRICS):
            
            if n_cases > 1 and n_metrics > 1:
                ax = axes[row_idx, col_idx]
            elif n_cases > 1:
                ax = axes[row_idx]
            elif n_metrics > 1:
                ax = axes[col_idx]
            else:
                ax = axes

            subset = df[df['Case'] == case]
            
            sns.barplot(
                data=subset,
                x='Condition',
                y=metric,
                hue='method',
                hue_order=METHOD_ORDER,
                palette=PALETTE,
                ax=ax,
                edgecolor='black',
                linewidth=0.5,
                errorbar='sd',
                capsize=0.1,
                err_kws={'linewidth': 1}
            )
            
            # Headers
            if row_idx == 0:
                ax.set_title(METRIC_LABELS[metric], fontsize=20, fontweight='bold', pad=15)
            else:
                ax.set_title("")

            # Row Labels (Case Names)
            if col_idx == 0:
                # No textwrap! The string already has newlines.
                # Using va='center' to align the 3 lines block with the plot center
                # labelpad moves it left.
                ax.set_ylabel(case, fontsize=18, rotation=0, labelpad=90, va='center')
            else:
                ax.set_ylabel("")

            if row_idx == n_cases - 1:
                ax.set_xlabel(r"Condition $(1_A,1_B)$", fontsize=18)
            else:
                ax.set_xlabel("")

            ax.grid(True, axis='y', linestyle='--', alpha=0.6)
            sns.despine(ax=ax, left=True)
            
            if ax.get_legend():
                ax.get_legend().remove()

    # Global Legend
    handles, labels = (axes[0, 0] if n_cases > 1 and n_metrics > 1 else axes[0]).get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=3, 
               bbox_to_anchor=(0.55, -0.01), fontsize=18, frameon=False)

    plt.tight_layout()
    
    # Adjust Margins
    # Increased left margin (0.2 -> 0.25) for the wider 3-line labels
    plt.subplots_adjust(top=0.92, bottom=0.07, left=0.25)
    
    # Add "Schedules" Header
    # Placed in the top-left margin area
    # x=0.125 is roughly centered in the 0.25 left margin
    fig.text(0.17, 0.93, "Schedules", fontsize=20, fontweight='bold', ha='center', va='bottom', fontfamily='Times New Roman')
    
    output_path = os.path.join(PARENT_DIR, OUTPUT_FILENAME)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Success! Plot saved to: {output_path}")
    plt.show()

# ==========================================
# 3. PRINT STATS TABLE
# ==========================================
def print_markdown_stats(df):
    print("\n" + "="*80)
    print(" COMPACT SUMMARY STATISTICS (Mean ± Std)")
    print("="*80 + "\n")
    
    # Create a clean copy for display
    df_disp = df.copy()
    
    # Clean up the LaTeX labels for the text table (Remove $, \alpha, newlines)
    # We want a single line string for the Case column
    def clean_label(s):
        # Replace newlines with spaces
        s = s.replace('\n', ' | ')
        # Remove LaTeX specific chars for cleaner text output
        s = s.replace('$', '').replace('\\', '')
        return s

    df_disp['Case'] = df_disp['Case'].apply(clean_label)

    # Group by Case, Condition, Method
    grouped = df_disp.groupby(['Case', 'Condition', 'method'])[METRICS].agg(['mean', 'std'])
    
    # Create output format "Mean ± Std"
    summary_df = pd.DataFrame(index=grouped.index)
    for metric in METRICS:
        summary_df[metric] = grouped[metric].apply(
            lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1
        )
    
    summary_df = summary_df.reset_index()
    
    # Try to use markdown if available, else string
    try:
        print(summary_df.to_markdown(index=False))
    except ImportError:
        print(summary_df.to_string(index=False))
    
    # Also save to CSV for easy copy-paste
    stats_path = os.path.join(PARENT_DIR, "Summary_Stats_Table.csv")
    summary_df.to_csv(stats_path, index=False)
    print(f"\n[Info] Stats saved to: {stats_path}")

if __name__ == "__main__":
    if os.path.exists(PARENT_DIR):
        combined_df = load_all_data(PARENT_DIR)
        if not combined_df.empty:
            create_facet_plot(combined_df)
            print_markdown_stats(combined_df)
    else:
        print(f"Error: Directory '{PARENT_DIR}' not found.")

## Sensitivity to the bump parameter $B$

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_6")
print(f"Experiment directory: {exp_dir}")

In [ ]:
# Multiple Bumps
ANNEAL_WEIGHT = 1.0
BUMP_VALUES = [10.0, 20.0, 30.0, 40.0, 50.0, 100.0]
RUN_METRIC_EVAL = True
names = ["cos_t", "sigmoid", "1-t**2"]
print(names)

for BUMP_VALUE in BUMP_VALUES:
    gamma_list = [
        lambda t : torch.tensor(1) * ANNEAL_WEIGHT+ (BUMP_VALUE * t * (1 - t)),
        lambda t : torch.tensor(-1) * ANNEAL_WEIGHT ,
        lambda t : torch.tensor(1)
    ]
    d_gamma_list = [
        lambda t: torch.zeros_like(t)+ (BUMP_VALUE * (1 - 2*t)),
        lambda t: torch.zeros_like(t) ,
        lambda t: torch.zeros_like(t)
    ]
    Criterion = lambda t: sum([ gamma_list[i](t) / (interpolant_schedules[names[i]].alpha_t(t))**2 for i in range(len(names)) ])

    print(f"Criterion(0.99) = {Criterion(torch.tensor(0.99)).item()}")

    t = torch.linspace(0.0, 0.99, 100)
    plt.plot(t.numpy(), Criterion(t).numpy(), label=f'Bump={BUMP_VALUE}')

heuristic = lambda t : 1 / (interpolant_schedules[names[0]].alpha_t(t))**2
t = torch.linspace(0.0, 0.99, 100)
plt.plot(t.numpy(), heuristic(t).numpy(), label='Bump Guide', linestyle='--')
plt.xlabel('t')
plt.ylabel('C(t)')
plt.title('Criterion C(t) for Multiple Bumps')
plt.grid(True)
plt.ylim(-10,40)
plt.legend()
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUES}.png"))
plt.rcParams['figure.dpi'] = 300
plt.show()

## Figure C.1: Performance comparison across varying ESS thresholds

In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_C1")
print(f"Experiment directory: {exp_dir}")

In [ ]:
names = ['cos_t', 'ddpm_linear', 'default_linear',0]

# for names in collapse_combinations:
for ESS_THRESHOLD in [0.1,0.3,0.5,0.7,0.9]:
    print(f"ESS Threshold: {ESS_THRESHOLD}")
    bs = 10000; n_steps=1000
    seeds = [0,1,2,3,4]
    ANNEAL_WEIGHT = 1.0
    BUMP_VALUE = 0.0
    RUN_METRIC_EVAL = True
    results = []

    subexperiment_id = f"{names}"
    experiment_id = f"{exp_dir}/[{names[3]:.4f}]{subexperiment_id}_ESS{ESS_THRESHOLD}_Noncollapse"
    if not os.path.exists(experiment_id):
        os.makedirs(experiment_id)

    u_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); u_model1.load_state_dict(torch.load(f"PretrainedToyModels/u_model1_X_given_A_alpha={names[0]}.pth")); u_model1.eval()
    s_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); s_model1.load_state_dict(torch.load(f"PretrainedToyModels/s_model1_X_given_A_alpha={names[0]}.pth")); s_model1.eval()
    u_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); u_model2.load_state_dict(torch.load(f"PretrainedToyModels/u_model2_XY_given_B_alpha={names[2]}.pth")); u_model2.eval()
    s_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); s_model2.load_state_dict(torch.load(f"PretrainedToyModels/s_model2_XY_given_B_alpha={names[2]}.pth")); s_model2.eval()
    u_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); u_model3.load_state_dict(torch.load(f"PretrainedToyModels/u_model3_X_alpha={names[1]}.pth")); u_model3.eval()
    s_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); s_model3.load_state_dict(torch.load(f"PretrainedToyModels/s_model3_X_alpha={names[1]}.pth")); s_model3.eval()

    def v1_fn(x, t, A): return u_model1(x, t, A)
    def s1_fn(x, t, A): return s_model1(x, t, A)
    def v2_fn(x, t): return u_model3(x, t)
    def s2_fn(x, t): return s_model3(x, t)
    def v3_fn(z, t, B): return u_model2(z, t, B)
    def s3_fn(z, t, B): return s_model2(z, t, B)
    def sigma_fn(t): return 0.5 * torch.ones_like(t)

    v_fn_list=[
            lambda x, t: v1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # v1(X|A)
            lambda x, t: v2_fn(x[:, :1], t),                                                 # v2(X)
            lambda x, t: v3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # v3(Z|B)
        ]
    s_fn_list=[
            lambda x, t: s1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # s1(X|A)
            lambda x, t: s2_fn(x[:, :1], t),                                                 # s2(X)
            lambda x, t: s3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # s3(Z|B)
        ]
    proj_list=[
            lambda z: z[:, :1],    # project to X 
            lambda z: z[:, :1],    # project to X
            lambda z: z            # identity for Z
        ]
    emb_list=[
            lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
            lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
            lambda z: z  # identity
        ]
    print(f"{names} models loaded.")


    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        print(f"Seed set to {seed}")

        for Method_name in ["NR", "ACE"]: # FKC:
            print(f"Method: {Method_name}")
            if Method_name == "FKC" or Method_name == "NR":
                    print("Simulating FKC (Constant Gammas)")
                    gamma_list = [
                        lambda t : torch.tensor(1) * ANNEAL_WEIGHT,
                        lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                        lambda t : torch.tensor(1)
                    ]
                    d_gamma_list = [
                        lambda t: torch.zeros_like(t),
                        lambda t: torch.zeros_like(t),
                        lambda t: torch.zeros_like(t)
                    ]
            elif Method_name == "ACE":
                print("Simulating ACE (Adaptive Gammas)")
                gamma_list = [
                    lambda t : torch.tensor(1) * ANNEAL_WEIGHT + (BUMP_VALUE * t * (1 - t)),
                    lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                    lambda t : torch.tensor(1)
                ]
                d_gamma_list = [
                    lambda t: torch.zeros_like(t) + (BUMP_VALUE * (1 - 2*t)),
                    lambda t: torch.zeros_like(t),
                    lambda t: torch.zeros_like(t)
                ]
            if seed == 0:
                Criterion = lambda t: sum([ gamma_list[i](t) / (interpolants[names[i]](t))**2 for i in range(len(names)-1) ])
                t = torch.linspace(0.0, 0.99, 100)
                plt.plot(t.numpy(), Criterion(t).numpy())
                plt.xlabel('t')
                plt.ylabel('Criterion C(t)')
                plt.title('Criterion C(t) vs t')
                plt.grid(True)
                plt.ylim(-20,100)
                plt.savefig(os.path.join(experiment_id, f"Criterion_plot_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUE}_Method={Method_name}.png"))
                plt.show()
                plt.close()

            # print when Criterion = 0
            for i in range(len(t)-1):
                if Criterion(t[i]) > 0 and Criterion(t[i+1]) < 0:
                    print("Criterion = 0 at t =", t[i].item())
                    break
            for i in range(len(t)-1):
                if Criterion(t[i]) < 0 and Criterion(t[i+1]) > 0:
                    print("Criterion = 0 at t =", t[i].item())
                    break
            for A, B in [(1,1)]: #[(1,1), (1,0), (0,1), (0,0)]:
                print(f"Conditioning on A={A}, B={B}")

                x0 = torch.randn(bs, 2).to("cuda")
                samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
                    x0=x0, v_fn_list=v_fn_list, s_fn_list=s_fn_list, proj_list=proj_list, emb_list=emb_list, sigma_fn=sigma_fn,
                    v_star= lambda z, t: v3_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
                    t0=0.0, t1=1.0, n_steps=n_steps, device="cuda", ess_threshold=ESS_THRESHOLD, print_resample_history=True,
                    gamma_list=gamma_list,
                    d_gamma_list=d_gamma_list,
                    resample= (Method_name != "NR")
                )
                samples = samples.cpu().numpy()
                if seed == 0:
                    plot_diagnostics(samples, logw_final, logw_history, save_name=f"{experiment_id}/alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}")
                    plot_path_trajectories(sample_history, n_frame=6, resample_history=None, experiment_id=experiment_id, name=f"alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}", deg=-50)
                    plt.close(); clear_output()

                if RUN_METRIC_EVAL:
                    print(f"Evaluating {Method_name} for AB={A}{B}")
                    samples_gt = ground_truth_hcg(bs, cond_A=A, cond_B=B).numpy()
                    w1, w2, mmd_rbf, total_var = compute_sample_based_metrics(
                        torch.tensor(samples_gt), torch.tensor(samples)
                    )
                    results.append([seed, Method_name, w1, w2, mmd_rbf, total_var, A, B, ESS_THRESHOLD])

                    df = pd.DataFrame(results, columns=["seed", "method", "W1", "W2", "MMD_RBF", "Total Var.", "A", "B", "ESS_Threshold"])
                    df.to_csv(f"{experiment_id}/experiment_results_numseeds{len(seeds)}_bs{bs}_n_steps{n_steps}_ESS{ESS_THRESHOLD}_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUE}.csv", index=False)

In [ ]:
import pandas as pd
# seaborn is installed from this experiment's pyproject.toml and uv.lock.
import seaborn as sns
import matplotlib.pyplot as plt
import io

# ==========================================
# 1. LOAD DATA (You can run the previous cell to generate this data)
# ==========================================
csv_data = """seed,method,W1,W2,MMD_RBF,Total Var.,A,B,ESS_Threshold
0,FKC,2.671122276924688,2.958711624716925,1.267066478729248,0.9104000000000001,1,1,0.1
0,ACE,0.45108392028256944,0.5294060959511929,0.09096813201904297,0.6539,1,1,0.1
0,FKC,3.4695492583910754,3.7952742631138228,1.7368170022964478,0.9193060491493384,1,1,0.3
0,ACE,0.2837551553082389,0.39101061972767515,0.026133954524993896,0.6348877853424109,1,1,0.3
0,FKC,2.635403127283796,2.9104787290369014,1.4934245347976685,0.9286786924939467,1,1,0.5
0,ACE,0.4462359031293904,0.5690063408124068,0.06979155540466309,0.6533240506329114,1,1,0.5
0,FKC,2.5580212045630644,2.8336678031973097,1.5015771389007568,0.9128231367169639,1,1,0.7
0,ACE,0.3362470547979451,0.4870415253971988,0.032257080078125,0.6351821164232847,1,1,0.7
0,FKC,2.383411665109474,2.6509240935734875,2.8717684745788574,0.93428920889537,1,1,0.9
0,ACE,0.37758675580423,0.5337254428397956,0.03982508182525635,0.6391539055449713,1,1,0.9
1,FKC,4.2284852264504345,4.585312101147793,2.3089380264282227,0.9195476532302596,1,1,0.1
1,ACE,0.4316968612763876,0.5378348636978592,0.07085424661636353,0.6533617661816358,1,1,0.1
2,FKC,1.930929985235221,2.276854818614506,1.1867969036102295,0.8915554360353439,1,1,0.1
2,ACE,0.4297983665602693,0.4962915973295295,0.0813295841217041,0.6556402243589743,1,1,0.1
3,FKC,2.004467736415929,2.3474785539773624,1.4649248123168945,0.909593657086224,1,1,0.1
3,ACE,0.5242704349986455,0.7009764740865341,0.03785830736160278,0.6377822764552911,1,1,0.1
4,FKC,2.7904990279800783,3.064268541298036,1.2418321371078491,0.925,1,1,0.1
4,ACE,0.4819212066345991,0.5788725075984056,0.0894121527671814,0.6654519356460533,1,1,0.1
1,FKC,4.547946963459682,4.921675030264583,2.8481316566467285,0.9297020820575628,1,1,0.3
1,ACE,0.2236679396480153,0.3075014204955778,0.018775463104248047,0.6322165849264338,1,1,0.3
2,FKC,2.0499435269531423,2.3533058824504858,1.586472511291504,0.9040960521121201,1,1,0.3
2,ACE,0.3237596034523107,0.4695252829295255,0.030008435249328613,0.6353644657863146,1,1,0.3
3,FKC,2.0004597852714214,2.344475139888799,1.4729975461959839,0.9182887445887447,1,1,0.3
3,ACE,0.25836356265242777,0.3714566058171081,0.020724594593048096,0.6320083083083083,1,1,0.3
4,FKC,4.084050908036088,4.450108195686759,2.4135093688964844,0.9208823529411765,1,1,0.3
4,ACE,0.24392678933833567,0.32033378496137577,0.021965444087982178,0.6313572864321608,1,1,0.3
1,FKC,4.565417691592315,4.939379009426702,2.8578455448150635,0.9235714372346877,1,1,0.5
1,ACE,0.347686367670207,0.48201250023863923,0.03843808174133301,0.6384377964575203,1,1,0.5
2,FKC,2.0269332292908295,2.370237978907089,1.1330214738845825,0.9078942710560067,1,1,0.5
2,ACE,0.37930631694566846,0.5050826542387004,0.04530757665634155,0.6368671679197995,1,1,0.5
3,FKC,1.987620273176426,2.3376318451993527,1.4207051992416382,0.9090190766108575,1,1,0.5
3,ACE,0.3156302133448867,0.451838641030446,0.02883732318878174,0.6356397111913357,1,1,0.5
4,FKC,4.680488438897764,5.0621037885062945,3.112687349319458,0.9286780976220275,1,1,0.5
4,ACE,0.2686304438141428,0.3546417513669526,0.0268593430519104,0.6375121121121121,1,1,0.5
1,FKC,4.4482112878494116,4.815448088293516,2.644001007080078,0.9214455953016552,1,1,0.7
1,ACE,0.26522724981493073,0.36215999959056405,0.022831201553344727,0.6282914891489149,1,1,0.7
2,FKC,2.1187640176370395,2.467563381293868,1.0419046878814697,0.8853414075286417,1,1,0.7
2,ACE,0.34453142230727757,0.4992698123291965,0.034919679164886475,0.6449338737115982,1,1,0.7
3,FKC,1.9767251619131003,2.3259969709536645,1.4283051490783691,0.9098335427742871,1,1,0.7
3,ACE,0.27452820674935186,0.3820974920280395,0.023885250091552734,0.6295593890836255,1,1,0.7
4,FKC,4.665837171434316,5.047875703377264,3.082986354827881,0.9192302904564315,1,1,0.7
4,ACE,0.3195528769562292,0.429236009123164,0.033013999462127686,0.6304146031428285,1,1,0.7
1,FKC,4.065684461611906,4.411413775700504,2.0091426372528076,0.9173573382430299,1,1,0.9
1,ACE,0.2973141426818991,0.4007880775002392,0.02447342872619629,0.6309641962944417,1,1,0.9
2,FKC,1.98012851415699,2.3221128230509764,1.3376667499542236,0.8948727272727273,1,1,0.9
2,ACE,0.2671305754103127,0.3840582329066812,0.022841036319732666,0.6349198278450605,1,1,0.9
3,FKC,1.9941951706572165,2.335591893648943,1.3994040489196777,0.9083838741396264,1,1,0.9
3,ACE,0.23806422919997175,0.3010554298584418,0.021109402179718018,0.6375040844929423,1,1,0.9
4,FKC,4.713815317336253,5.092829365058878,3.1451077461242676,0.9236224852071007,1,1,0.9
4,ACE,0.29863210080212105,0.39866761067141593,0.027861177921295166,0.6401947817360762,1,1,0.9
"""

df = pd.read_csv(io.StringIO(csv_data))

# ==========================================
# 2. CONFIG & STYLE
# ==========================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
sns.set_context("paper", font_scale=1.4)
sns.set_style("whitegrid")

METRIC_LABELS = {
    'W1': r'$W_1$ Distance ($\downarrow$)',
    'W2': r'$W_2$ Distance ($\downarrow$)',
    'MMD_RBF': r'MMD-RBF ($\downarrow$)',
    'Total Var.': r'Total Var. ($\downarrow$)'
}

PALETTE = {
    'FKC': '#696969',  # Dark Gray
    'ACE': '#D62728'   # Bold Red
}

# ==========================================
# 3. PLOT GENERATION
# ==========================================
def plot_ess_sensitivity(df):
    # Melt for plotting
    df_melted = df.melt(
        id_vars=['ESS_Threshold', 'method'], 
        value_vars=['W1', 'W2', 'MMD_RBF', 'Total Var.'],
        var_name='Metric', value_name='Score'
    )

    # Create Grid
    g = sns.FacetGrid(df_melted, col='Metric', hue='method', 
                      sharey=False, height=4, aspect=1.0, palette=PALETTE)
    
    g.map(sns.lineplot, 'ESS_Threshold', 'Score', marker='o', linewidth=2.5, markersize=8)

    # Customize
    for ax, title in zip(g.axes.flat, ['W1', 'W2', 'MMD_RBF', 'Total Var.']):
        ax.set_title(METRIC_LABELS[title], fontweight='bold', fontsize=14, pad=15)
        ax.set_xlabel(r"ESS Threshold ($\tau$)", fontsize=12)
        
        # Highlight 0.7 as the chosen operating point
        ax.axvline(x=0.7, color='black', linestyle='--', alpha=0.3)

    # Add Legend manually to ensure placement
    g.add_legend(title="Method", fontsize=12, title_fontsize=12)
    
    plt.subplots_adjust(top=0.85)
    # g.fig.suptitle("Sensitivity to Resampling Threshold (ESS)", 
                #    fontsize=16, fontweight='bold', fontfamily='serif')
    
    plt.savefig("ESS_Sensitivity_Plot.png", dpi=300, bbox_inches='tight')
    print("Plot saved to ESS_Sensitivity_Plot.png")
    plt.show()

# ==========================================
# 4. LATEX TABLE GENERATION
# ==========================================
def generate_latex_table(df):
    # Pivot to get side-by-side columns for methods
    pivot_df = df.pivot(index='ESS_Threshold', columns='method', values=['W1', 'W2', 'MMD_RBF', 'Total Var.'])
    
    print("\n=== LaTeX Table Code ===")
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Performance comparison across varying ESS Thresholds. \textbf{Bold} indicates the best method.}")
    print(r"\label{tab:ess_ablation}")
    print(r"\begin{tabular}{c|cc|cc|cc}")
    print(r"\toprule")
    print(r"ESS & \multicolumn{2}{c|}{$W_1 (\downarrow)$} & \multicolumn{2}{c|}{$W_2 (\downarrow)$} & \multicolumn{2}{c}{MMD (\downarrow)$} \\")
    print(r"Threshold & FKC & ACE & FKC & ACE & FKC & ACE \\")
    print(r"\midrule")

    for ess in pivot_df.index:
        row = pivot_df.loc[ess]
        line = f"{ess:.1f}"
        
        # We only showing W1, W2, MMD for brevity in table (TV is often omitted in main text tables)
        # You can add 'Total Var.' back if needed
        for metric in ['W1', 'W2', 'MMD_RBF']:
            fkc_val = row[(metric, 'FKC')]
            ace_val = row[(metric, 'ACE')]
            
            fkc_str = f"{fkc_val:.3f}"
            ace_str = f"{ace_val:.3f}"
            
            if fkc_val < ace_val:
                fkc_str = f"\\textbf{{{fkc_str}}}"
            else:
                ace_str = f"\\textbf{{{ace_str}}}"
            
            line += f" & {fkc_str} & {ace_str}"
        
        line += r" \\"
        print(line)
        
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

# Run
plot_ess_sensitivity(df)
# generate_latex_table(df)

## Figure E.5: Path exitence criterion $C(t)$ across various schedule combinations


In [ ]:
exp_dir = get_experiment_dir(f"ace_demo_runs_{datetime.now().strftime('%Y%m%d')}", "figure_E5")
print(f"Experiment directory: {exp_dir}")

In [ ]:
ANNEAL_WEIGHT = 1.0 #15


model_path = "PretrainedToyModels"
interpolants = load_interpolants_from_json_alpha_only("ace_lib/interpolant_schedules.json")
names = list(interpolants.keys())

t = torch.linspace(0.0, 1.0, 100)

name_eq = {
    "ddpm_linear": r"$\alpha_t = \text{DDPM}$",
    "1-t**2": r"$\alpha_t = 1-t^2$",
    "sigmoid": r"$\alpha_t = \text{Sigmoid}$",
    "default_linear": r"$\alpha_t = 1-t$",
    "cos_t": r"$\alpha_t=\cos(\frac{\pi}{2}t)$"
}

def check_condition(alpha_funcs, n_grid=200, Bump = 0.0, Anneal_weight=1.0):
    """Check sign conditions for [a1, a2, a3]. a1 a3 / a2 and anneal weight applies to (a1 / a2)^w a3"""
    ts = torch.linspace(0.0, 0.99, n_grid)

    if n_grid > 1:
        dt = ts[1] - ts[0]
    else:
        dt = torch.tensor(0.0) 

    alphas = [f(ts) for f in alpha_funcs]
    Bumps = torch.tensor([Bump * t * (1-t) for t in ts])
    if len(alpha_funcs) == 3:
        C = (Anneal_weight + Bumps) / (alphas[0]**2 + 1e-12) - Anneal_weight / (alphas[1]**2 + 1e-12) + 1 / (alphas[2]**2 + 1e-12)
    else:
        raise ValueError("Only supports 3 schedules")

    total_negative_length = 0.0
    if n_grid > 1:
        negative_intervals = C[:-1] < 0
        total_negative_length_tensor = torch.sum(negative_intervals.float()) * dt
        total_negative_length = total_negative_length_tensor.item()

    return (C.min() < 0), ts, C, total_negative_length

name_eq_plot = {
    "ddpm_linear": r"$\text{DDPM}$",
    "1-t**2": r"$1-t^2$",
    "sigmoid": r"$\text{Sigmoid}$",
    "default_linear": r"$1-t$",
    "cos_t": r"$\cos(\frac{\pi}{2}t)$"
}


def find_valid_combinations(interpolants, Anneal_weight=1.0, Bump=0.0, unique=True):
    collapse_combinations = []

    for a1, a2, a3 in itertools.product(interpolants, repeat=3):
        valid, t, C, total_negative_length = check_condition([interpolants[a1], interpolants[a2], interpolants[a3]], Anneal_weight=Anneal_weight, Bump=Bump)
        if valid:
            collapse_combinations.append([a1, a2, a3, total_negative_length])
    collapse_combinations.sort(key=lambda x: x[3], reverse=True)
    if unique:
        unique_combinations = []
        seen = set()
        for combo in collapse_combinations:
            identifier = combo[3]
            if identifier not in seen:
                unique_combinations.append(combo)
                seen.add(identifier)
        collapse_combinations = unique_combinations # Remove duplicates if total_negative_length is the same
    return collapse_combinations




collapse_combinations = find_valid_combinations(interpolants, Anneal_weight=ANNEAL_WEIGHT, Bump=0.0, unique=False)
print(f"There are {len(collapse_combinations)} combinations that have path collapse:")
with open(os.path.join(exp_dir, "non_unique_collapse_combinations.json"), "w") as f:
    json.dump(collapse_combinations, f, indent=4)

from tqdm import tqdm
plt.figure(figsize=(15,12))
plt.title(f'Criterion C(t) with ACE Correction Three-Expert Composition with w={ANNEAL_WEIGHT}', fontsize=24)
for names in tqdm(collapse_combinations):
    a1, a2, a3, invalid_interval = names
    valid = False
    Bump = 50.0
    while not valid:
        valid, t, C, invalid_interval = check_condition([interpolants[a1], interpolants[a2], interpolants[a3]], Anneal_weight=ANNEAL_WEIGHT, Bump=Bump)
        exceed_indices = torch.where(C > 200)[0]
        if len(exceed_indices) > 0:
            first_exceed_index = exceed_indices[0]
            C = C[:first_exceed_index]
            t = t[:first_exceed_index]
        valid = C.min() >= 0
        Bump = Bump + 70
    plt.ylim((-20,100))
    plt.grid(True)
    plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
    plt.legend(loc='upper left', fontsize=12, ncol=4)
plt.xlabel('t', fontsize=24)
plt.ylabel(r"$C(t)$", fontsize=24)
plt.tight_layout()
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_ACE_w={ANNEAL_WEIGHT}.png"))
plt.show()
plt.close()
clear_output()

plt.figure(figsize=(15,12))
plt.title(f'Criterion C(t) with Constant Exponents for Three-Expert Composition with w={ANNEAL_WEIGHT}', fontsize=24)
for names in tqdm(collapse_combinations):
    a1, a2, a3, invalid_interval = names
    valid = False
    valid, t, C, invalid_interval = check_condition([interpolants[a1], interpolants[a2], interpolants[a3]], Anneal_weight=ANNEAL_WEIGHT, Bump=0.0)
    plt.ylim((-20,100))
    plt.grid(True)
    plt.plot(t.numpy(), C.numpy(), label=f'{name_eq_plot[a1]}, {name_eq_plot[a2]}, {name_eq_plot[a3]}')
    plt.legend(loc='upper left', fontsize=12, ncol=4)
plt.xlabel('t', fontsize=24)
plt.ylabel(r"$C(t)$", fontsize=24)
plt.tight_layout()
plt.savefig(os.path.join(exp_dir, f"Criterion_plot_FKC_w={ANNEAL_WEIGHT}.png"))
plt.show()
plt.close()

## Figure E.6: Quantitative evaluation of Heterogeneous Ratio-of-Densities Sampling

Using the generated plots, we can can also make Figures E.7~E.11: Visualization of generative trajectories and final samples.

In [ ]:
# The following loop may take a long time to run for all collapse combinations and seeds. 
# We report five randomly selected collapse combinations in our paper:
collapse_combinations = [["ddpm_linear", "default_linear", "ddpm_linear", 7.5],
                         ["cos_t", "default_linear", "ddpm_linear", 8.9],
                         ["ddpm_linear", "1-t**2", "default_linear", 10.9],
                         ["cos_t", "sigmoid", "cos_t", 11.4],
                         ["cos_t", "default_linear", "cos_t", 48.8]]

for names in collapse_combinations:
    bs = 10000; n_steps=1000
    seeds = [0,1,2,3,4]
    ESS_THRESHOLD = 0.7
    ANNEAL_WEIGHT = 1.0
    BUMP_VALUE = 30.0
    RUN_METRIC_EVAL = True
    results = []

    subexperiment_id = f"{names}"
    experiment_id = f"{exp_dir}/[{names[3]:.4f}]{subexperiment_id}_Visualizations"
    if not os.path.exists(experiment_id):
        os.makedirs(experiment_id)

    u_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); u_model1.load_state_dict(torch.load(f"{model_path}/u_model1_X_given_A_alpha={names[0]}.pth")); u_model1.eval()
    s_model1 = MLPInstFlexible(z_dim=1, cond_dim=1, width=256, depth=4, output_dim=1).to(device); s_model1.load_state_dict(torch.load(f"{model_path}/s_model1_X_given_A_alpha={names[0]}.pth")); s_model1.eval()
    u_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); u_model2.load_state_dict(torch.load(f"{model_path}/u_model2_XY_given_B_alpha={names[2]}.pth")); u_model2.eval()
    s_model2 = MLPInstFlexible(z_dim=2, cond_dim=1, width=256, depth=4, output_dim=2).to(device); s_model2.load_state_dict(torch.load(f"{model_path}/s_model2_XY_given_B_alpha={names[2]}.pth")); s_model2.eval()
    u_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); u_model3.load_state_dict(torch.load(f"{model_path}/u_model3_X_alpha={names[1]}.pth")); u_model3.eval()
    s_model3 = MLPInstFlexible(z_dim=1, cond_dim=0, width=256, depth=4, output_dim=1).to(device); s_model3.load_state_dict(torch.load(f"{model_path}/s_model3_X_alpha={names[1]}.pth")); s_model3.eval()

    def v1_fn(x, t, A): return u_model1(x, t, A)
    def s1_fn(x, t, A): return s_model1(x, t, A)
    def v2_fn(x, t): return u_model3(x, t)
    def s2_fn(x, t): return s_model3(x, t)
    def v3_fn(z, t, B): return u_model2(z, t, B)
    def s3_fn(z, t, B): return s_model2(z, t, B)
    def sigma_fn(t): return 0.5 * torch.ones_like(t)

    v_fn_list=[
            lambda x, t: v1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # v1(X|A)
            lambda x, t: v2_fn(x[:, :1], t),                                                 # v2(X)
            lambda x, t: v3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # v3(Z|B)
        ]
    s_fn_list=[
            lambda x, t: s1_fn(x[:, :1], t, torch.full((x.size(0), 1), A, device=x.device)), # s1(X|A)
            lambda x, t: s2_fn(x[:, :1], t),                                                 # s2(X)
            lambda x, t: s3_fn(x, t, torch.full((x.size(0), 1), B, device=x.device))         # s3(Z|B)
        ]
    proj_list=[
            lambda z: z[:, :1],    # project to X 
            lambda z: z[:, :1],    # project to X
            lambda z: z            # identity for Z
        ]
    emb_list=[
            lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
            lambda x: torch.cat([x, torch.zeros(x.size(0), 1, device=x.device)], dim=1),  # embed X→Z
            lambda z: z  # identity
        ]
    print(f"{names} models loaded.")


    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        print(f"Seed set to {seed}")

        for Method_name in ["FKC", "ACE"]:
            print(f"Method: {Method_name}")
            if Method_name == "FKC":
                    print("Simulating FKC (Constant Gammas)")
                    gamma_list = [
                        lambda t : torch.tensor(1) * ANNEAL_WEIGHT,
                        lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                        lambda t : torch.tensor(1)
                    ]
                    d_gamma_list = [
                        lambda t: torch.zeros_like(t),
                        lambda t: torch.zeros_like(t),
                        lambda t: torch.zeros_like(t)
                    ]
            elif Method_name == "ACE":
                print("Simulating ACE (Adaptive Gammas)")
                gamma_list = [
                    lambda t : torch.tensor(1) * ANNEAL_WEIGHT + (BUMP_VALUE * t * (1 - t)),
                    lambda t : torch.tensor(-1) * ANNEAL_WEIGHT,
                    lambda t : torch.tensor(1)
                ]
                d_gamma_list = [
                    lambda t: torch.zeros_like(t) + (BUMP_VALUE * (1 - 2*t)),
                    lambda t: torch.zeros_like(t),
                    lambda t: torch.zeros_like(t)
                ]

            for A, B in [(1,1), (1,0), (0,1), (0,0)]:
                print(f"Conditioning on A={A}, B={B}")

                x0 = torch.randn(bs, 2).to("cuda")
                samples, logw_final, logw_history, sample_history, resample_history = simulate_ace(
                    x0=x0, v_fn_list=v_fn_list, s_fn_list=s_fn_list, proj_list=proj_list, emb_list=emb_list, sigma_fn=sigma_fn,
                    v_star= lambda z, t: v3_fn(z, t, torch.full((z.size(0), 1), B, device=z.device)), 
                    t0=0.0, t1=1.0, n_steps=n_steps, device="cuda", ess_threshold=ESS_THRESHOLD, print_resample_history=True,
                    gamma_list=gamma_list,
                    d_gamma_list=d_gamma_list,
                    resample=True
                )
                samples = samples.cpu().numpy()
                if seed == 0:
                    plot_diagnostics(samples, logw_final, logw_history, save_name=f"{experiment_id}/alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}")
                    plot_path_trajectories(sample_history, n_frame=6, resample_history=None, experiment_id=experiment_id, name=f"alpha={names}_{Method_name}_seed{seed}_AB={A}{B}_Bump={BUMP_VALUE}", deg=-50)
                    plt.close(); clear_output()

                if RUN_METRIC_EVAL:
                    print(f"Evaluating {Method_name} for AB={A}{B}")
                    samples_gt = ground_truth_hcg(bs, cond_A=A, cond_B=B).numpy()
                    w1, w2, mmd_rbf, total_var = compute_sample_based_metrics(
                        torch.tensor(samples_gt), torch.tensor(samples)
                    )
                    results.append([seed, Method_name, w1, w2, mmd_rbf, total_var, A, B])

                    df = pd.DataFrame(results, columns=["seed", "method", "W1", "W2", "MMD_RBF", "Total Var.", "A", "B"])
                    df.to_csv(f"{experiment_id}/experiment_results_numseeds{len(seeds)}_bs{bs}_n_steps{n_steps}_ESS{ESS_THRESHOLD}_{names}_ANNEAL={ANNEAL_WEIGHT}_BUMP={BUMP_VALUE}.csv", index=False)

In [ ]:
import os
import glob
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast

# ==========================================
# CONFIGURATION
# ==========================================
PARENT_DIR = exp_dir
OUTPUT_FILENAME = "fig_e6.png"
plt.rcParams['savefig.dpi'] = 300 # For saved figure resolution

# Metrics to plot (Columns)
METRICS = ['W1', 'W2', 'MMD_RBF']
METRIC_LABELS = {
    'W1': r'$W_1$ Distance ($\downarrow$)',
    'W2': r'$W_2$ Distance ($\downarrow$)',
    'MMD_RBF': r'MMD-RBF ($\downarrow$)'
}

# LaTeX Mappings for Schedules
NAME_EQ_PLOT = {
    "ddpm_linear": r"$\text{DDPM}$",
    "1-t**2": r"$1-t^2$",
    "sigmoid": r"$\text{Sigmoid}$",
    "default_linear": r"$1-t$",
    "cos_t": r"$\cos(\frac{\pi}{2}t)$"
}

# Methods and Colors (Highlight ACE)
PALETTE = {
    'NR': '#B0B0B0',   # Light Gray
    'FKC': '#696969',  # Dark Gray
    'ACE': '#D62728'   # Bold Red
}

# Order of plotting on X-axis
METHOD_ORDER = ['NR', 'FKC', 'ACE']

# ==========================================
# 1. DATA LOADING
# ==========================================
def load_all_data(parent_dir):
    all_data = []
    
    # 1. ROBUST FOLDER FINDING
    try:
        subdirs = [
            os.path.join(parent_dir, d) for d in os.listdir(parent_dir) 
            if os.path.isdir(os.path.join(parent_dir, d)) and d.endswith("_Visualizations")
        ]
    except FileNotFoundError:
        print(f"Error: Parent directory '{parent_dir}' not found.")
        return pd.DataFrame() 
    
    # Sort them based on the float number in the first bracket
    def sort_key(path):
        match = re.search(r"\[([\d\.]+)\]", os.path.basename(path))
        return float(match.group(1)) if match else 0
    
    subdirs.sort(key=sort_key)

    for i, folder_path in enumerate(subdirs):
        folder_name = os.path.basename(folder_path)
        
        # Extract and parse the schedule list for the Case Label
        try:
            # The folder structure is usually: [sort_val]['s1', 's2', 's3', len]_Visualizations
            # We extract the list part between ']' and '_Visualizations'
            raw_list_part = folder_name.split(']')[1].split('_Visualizations')[0]
            
            if not raw_list_part.endswith(']'):
                raw_list_part += ']'
            
            # Parse list
            schedule_list = ast.literal_eval(raw_list_part)
            schedules = schedule_list[:3]
            schedule_collapse_length = schedule_list[3] if len(schedule_list) > 3 else "N/A"
            # convert string to float with 1 decimal place
            try:
                schedule_collapse_length = f"{float(schedule_collapse_length)*100:.1f}"
            except:
                schedule_collapse_length = "N/A"
            
            # 1. Map to LaTeX (strip existing $ so we can wrap it uniformly)
            latex_schedules = [NAME_EQ_PLOT.get(s, s).replace('$', '') for s in schedules]
            
            # 2. Apply Reordering Logic (User specified: [2, 0, 1])
            # Original: [s1, s2, s3] -> New: [s3, s1, s2]
            reordered = [latex_schedules[2]] + latex_schedules[:2]
            
            # 3. Format into 3 lines: alpha^(i)_t = val
            formatted_lines = []
            for idx, val in enumerate(reordered):
                # Construct LaTeX string: $\alpha^{(i)}_t = val$
                # Using raw strings to handle backslashes safely
                line = r"$\alpha^{(" + str(idx+1) + r")}_t = " + val + r"$"
                formatted_lines.append(line)
            formatted_lines.append(f"\nCollapse Duration: {schedule_collapse_length}%")
            case_label = "\n".join(formatted_lines)

        except Exception as e:
            print(f"Warning: Could not parse label for {folder_name}, using default. Error: {e}")
            case_label = f"Case {i+1}"

        # 2. ROBUST CSV FINDING
        try:
            files_in_folder = os.listdir(folder_path)
            csv_files = [
                os.path.join(folder_path, f) for f in files_in_folder
                if f.startswith("experiment_results") and f.endswith(".csv")
            ]
        except OSError:
            print(f"Could not access folder: {folder_name}")
            continue

        if not csv_files:
            print(f"Skipping {folder_name}: No results CSV found.")
            continue
            
        # Read Data
        df = pd.read_csv(csv_files[0])
        df['Condition'] = df.apply(lambda row: f"({int(row['A'])},{int(row['B'])})", axis=1)
        df['Case'] = case_label
        df['Case_Index'] = i 
        all_data.append(df)

    if not all_data:
        raise ValueError("No data found! Check your paths.")
        
    return pd.concat(all_data, ignore_index=True)

# ==========================================
# 2. PLOTTING
# ==========================================
def create_facet_plot(df):
    # --- FONT SETTINGS ---
    # Set global font family to serif
    plt.rcParams['font.family'] = 'Times New Roman'
    sns.set_context("paper", font_scale=2)

    cases = df.sort_values('Case_Index')['Case'].unique()
    n_cases = len(cases)
    n_metrics = len(METRICS)
    
    # Increase height per case to accommodate 3 lines of text
    # Width = 5 * n_metrics, Height = 3.5 * n_cases (was 3.0)
    fig, axes = plt.subplots(n_cases, n_metrics, figsize=(5 * n_metrics, 3.5 * n_cases), sharex=True)
    
    sns.set_style("whitegrid")

    print("Generating Plots...")

    for row_idx, case in enumerate(cases):
        for col_idx, metric in enumerate(METRICS):
            
            if n_cases > 1 and n_metrics > 1:
                ax = axes[row_idx, col_idx]
            elif n_cases > 1:
                ax = axes[row_idx]
            elif n_metrics > 1:
                ax = axes[col_idx]
            else:
                ax = axes

            subset = df[df['Case'] == case]
            
            sns.barplot(
                data=subset,
                x='Condition',
                y=metric,
                hue='method',
                hue_order=METHOD_ORDER,
                palette=PALETTE,
                ax=ax,
                edgecolor='black',
                linewidth=0.5,
                errorbar='sd',
                capsize=0.1,
                err_kws={'linewidth': 1}
            )
            
            # Headers
            if row_idx == 0:
                ax.set_title(METRIC_LABELS[metric], fontsize=20, fontweight='bold', pad=15)
            else:
                ax.set_title("")

            # Row Labels (Case Names)
            if col_idx == 0:
                # No textwrap! The string already has newlines.
                # Using va='center' to align the 3 lines block with the plot center
                # labelpad moves it left.
                ax.set_ylabel(case, fontsize=18, rotation=0, labelpad=90, va='center')
            else:
                ax.set_ylabel("")

            if row_idx == n_cases - 1:
                ax.set_xlabel(r"Condition $(1_A,1_B)$", fontsize=18)
            else:
                ax.set_xlabel("")

            ax.grid(True, axis='y', linestyle='--', alpha=0.6)
            sns.despine(ax=ax, left=True)
            
            if ax.get_legend():
                ax.get_legend().remove()

    # Global Legend
    handles, labels = (axes[0, 0] if n_cases > 1 and n_metrics > 1 else axes[0]).get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=3, 
               bbox_to_anchor=(0.55, -0.01), fontsize=18, frameon=False)

    plt.tight_layout()
    
    # Adjust Margins
    # Increased left margin (0.2 -> 0.25) for the wider 3-line labels
    plt.subplots_adjust(top=0.92, bottom=0.07, left=0.25)
    
    # Add "Schedules" Header
    # Placed in the top-left margin area
    # x=0.125 is roughly centered in the 0.25 left margin
    fig.text(0.17, 0.93, "Schedules", fontsize=20, fontweight='bold', ha='center', va='bottom', fontfamily='Times New Roman')
    
    output_path = os.path.join(PARENT_DIR, OUTPUT_FILENAME)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Success! Plot saved to: {output_path}")
    plt.show()

if __name__ == "__main__":
    if os.path.exists(PARENT_DIR):
        combined_df = load_all_data(PARENT_DIR)
        if not combined_df.empty:
            create_facet_plot(combined_df)
    else:
        print(f"Error: Directory '{PARENT_DIR}' not found.")